## Notebook 4 of 5 — Validate WBM Against OpenET & Calibrate

Aligns WBM, OpenET, and flux-tower ET on common units/columns, computes
bias/RMSE/NSE performance metrics, then attempts bias-correction approaches
for the Oudin-driven WBM run and re-runs the WBM with the calibrated
parameters. This feeds directly into the nationwide method comparison in
notebook 07: assessing PET/AET method applicability across the US, and
whether a simple calibration to OpenET is worth adding to Oudin's
temperature-only simplicity relative to leaving it uncalibrated or using
Penman-Monteith outright.

**Calibration now uses a held-out train/test split**, so the numbers this
notebook (and notebook 07) reports for the calibrated variants reflect
genuine out-of-sample skill, not an optimistic in-sample fit: `PET_mult` is
fit using only 2016-2021 (`CAL_START`-`CAL_END`); 2022-2023
(`TEST_START`-`TEST_END`) is never touched by the fit and is reserved for
evaluation in notebook 07. This matters because a calibrated method that
only looks good on the years it was tuned to isn't actually more trustworthy
than an uncalibrated one — restricting to a held-out window is how you'd
actually find out.

**Two calibration granularities, not one.** This notebook fits **both**:
a `PET_mult` per site (the original approach — captures local conditions
best, but doesn't generalize to a site the model hasn't seen, e.g. a new
park), and a single pooled `PET_mult` per ecosystem (new — one value shared
by every site of that ecosystem type, cheaper to apply to a new site and a
step closer to the eventual goal of picking one nationwide method, not a
best-per-location patchwork). Notebook 07 compares both of these against
default (uncalibrated) Oudin and default Penman-Monteith, on the held-out
test period, to see which one option is actually worth adopting nationwide —
this notebook does not make that call itself.

*Part of a 5-notebook pipeline (run in order; each caches its outputs to `Data/` so later notebooks can be re-run without repeating expensive GEE/download steps): `01_select_parks_towers` -> `02_load_flux_openet_gridmet` -> `03_run_wbm` -> `04_validate_and_calibrate` -> `05_figures_and_export`.*

<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2025/blob/main/lectures/lecture4-ET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CIVE 523 Final Project – Kristen Cognac, February 27, 2026

Objective: The National Park Service (NPS) Water Balance Model (WBM) employs a set of parameterizations and functions to estimate a daily water balance for components rain, snow, snowmelt, soil water storage, evapotranspiration, and lumped runoff (surface water and groundwater). Actual evapotranspiration (ETa) is estimated using a simple "bucket-type" approach wherein potential evapotranspiration (Oudin, 2005) is maximized to the extent of available soil moisture storage. The NPS WBM has been recently applied across parks to assess past and future changes in water availability (e.g., Thoma, 2019; Thoma, 2020). However, detailed validation of model components has yet to be conducted, and the WBM's default PET method has not been benchmarked nationwide against an alternative formulation or evaluated for whether a simple, temperature-based calibration could improve its accuracy.

OpenET, with daily estimates of ETa from six remote sensing models, provides a robust, spatially continuous dataset for validating NPS WBM ETa. This project evaluates NPS WBM ETa accuracy **across the full nationwide network of AmeriFlux/USGS flux tower sites with data overlapping the OpenET record (2016-2023)** -- not just sites near National Parks -- by comparing it to OpenET monthly and annual timeseries using common statistical metrics. Given that OpenET has errors too, OpenET accuracy will also be assessed using ground-truth measurements from flux towers. The implications of ETa inaccuracy on water availability assessments will be considered by comparing error magnitude to the total water balance at each location. This project relies heavily on previous data and analysis compiled by Volk et al. (2024), which assessed the accuracy of OpenET across CONUS. In particular, they provide corrected timeseries of in situ ETa (and OpenET) through Zenodo repositories that are useful for estimating OpenET accuracy (Volk et al., 2023a; 2023b).

Because Oudin's temperature-only formulation systematically under- or over-estimates ETa relative to OpenET at many sites, this project also tests whether a simple multiplicative PET calibration to OpenET can close that gap while preserving Oudin's key practical advantage over the alternative evaluated here (Penman-Monteith): Penman-Monteith requires real wind speed, which is not available from LOCA2-downscaled CMIP6 climate projections, so a Penman-Monteith-driven WBM cannot be scaled to future-climate water availability assessments the way a temperature-only method can. The ultimate goal is to identify a **single** PET/AET method -- default Oudin, a calibrated Oudin, or Penman-Monteith -- for the NPS WBM to use nationwide, rather than whichever method performs best locally at each individual site.

Methods: Eddy covariance towers (AmeriFlux, USGS NWSC) and OpenET are compared to NPS WBM ETa calculated using Oudin and Penman-Monteith. For flux tower comparisons, daily ETa (2016-2023) is calculated for the grid cell overlapping each flux tower point using 4km GridMET precipitation and temperature inputs to the NPS WBM. OpenET, with a 30m grid, is spatially averaged for each GridMET 4km² cell to generate daily, monthly, and annual ETa.

To test whether calibration improves Oudin's accuracy without overstating it, a multiplicative PET calibration factor is fit to OpenET using only a 2016-2021 calibration period, at two granularities: one factor optimized per site, and one pooled factor optimized per ecosystem type (grouping all sites of a given land-cover class into a single fit, a step closer to a single nationwide value than a site-by-site patchwork). Both calibrated variants, along with default (uncalibrated) Oudin and default Penman-Monteith, are then evaluated only on the 2022-2023 held-out period that no calibration was fit to, so reported accuracy reflects genuine out-of-sample skill rather than an optimistic in-sample fit.

NPS WBM ETa accuracy will be assessed using: linear regression slope (forced through the origin), mean bias error (MBE), mean absolute error (MAE), root-mean-square error (RMSE), and coefficient of determination (r²). Because a method that performs well on average but poorly at a subset of sites is a riskier choice to standardize on nationwide, robustness across the four candidate methods is assessed using the mean, median, and worst-decile (90th-percentile) RMSE across sites, rather than the mean alone. Flux tower data (Volk et al., 2023a) will be resampled to daily, monthly, and annual values and compared to the corresponding OpenET ETa pixel (Volk et al., 2023b). OpenET accuracy will be similarly assessed using linear regression, MBE, MAE, RMSE, and r². Relative error metrics (e.g., MAE/Precipitation) will be evaluated to understand impacts on water availability assessments.

References:

Oudin, L., Hervieu, F., Michel, C., Perrin, C., Andréassian, V., Anctil, F., & Loumagne, C. (2005). Which potential evapotranspiration input for a lumped rainfall–runoff model?: Part 2—Towards a simple and efficient potential evapotranspiration model for rainfall–runoff modelling. Journal of hydrology, 303(1-4), 290-306.

Thoma, D. P., Munson, S. M., & Witwicki, D. L. (2019). Landscape pivot points and responses to water balance in national parks of the southwest US. Journal of Applied Ecology, 56(1), 157-167.

Thoma, D. P., Tercek, M. T., Schweiger, E. W., Munson, S. M., Gross, J. E., & Olliff, S. T. (2020). Water balance as an indicator of natural resource condition: Case studies from Great Sand Dunes National Park and Preserve. Global Ecology and Conservation, 24, e01300.

John M. Volk, Justin L. Huntington, Forrest Melton, Blake Minor, Tianxin Wang, Saseendran S. Anapalli, Raymond G. Anderson, Steven R. Evett, Andrew N. French, Richard Jasoni, Nicolas Bambach, William P. Kustas, Joseph G. Alfieri, John Prueger, Lawrence Hipps, Lynn McKee, Sebastian J. Castro Bustamante, Maria del Mar Alsina, Andrew McElrone, … Martha Anderson. (2023a). Post-processed data and graphical tools for a CONUS-wide eddy flux evapotranspiration dataset (1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.7636781

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., Kilic, A., Ruhoff, A., Senay, G. B., Minor, B., Morton, C., Ott, T., Johnson, L., Andrade, B. C. D., Carrara, W., Doherty, C. T., Dunkerly, C., Friedrichs, M., Guzman, A., … Yang, Y. (2023b). OpenET model data for assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10119477

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., ... & Yang, Y. (2024). Assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications. Nature Water, 2(2), 193-205.





# Setup Workspace

Import necessary libraries.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMPORTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Make the repo root (parent of notebooks/) importable ─────────────────────
import sys
from pathlib import Path as _Path
_REPO_ROOT = _Path.cwd().parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# ── Standard library ──────────────────────────────────────────────────────────
import importlib
import os
import time
import tempfile
import warnings
import zipfile
from datetime import date
from pathlib import Path

# ── Scientific computing ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr

# ── Geospatial ────────────────────────────────────────────────────────────────
import geopandas as gpd
import rasterio
import shapely
from rasterio.transform import rowcol
from shapely.geometry import Point
from adjustText import adjust_text

# ── Google Earth Engine ───────────────────────────────────────────────────────
import ee
import geemap

# ── Visualization ─────────────────────────────────────────────────────────────
import branca.colormap as cm
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# ── Statistics & optimization ─────────────────────────────────────────────────
import requests
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import pdist, squareform

# ── Utilities ─────────────────────────────────────────────────────────────────
from adjustText import adjust_text
from tqdm import tqdm

# ── NPS WBM (local package, repo_root/wbm/) ───────────────────────────────────
import wbm
from wbm import (
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run
)


Authenticate Google Earth Engine

In [ ]:
#if not ee.data._credentials:
ee.Authenticate()
ee.Initialize(project='modis-475315')

Define helper functions

In [ ]:
# functions needed for this workflow

# this function is used to add a google earth engine layer to an existing folium map,
# for visualization purposes. Folium is a python package that can put rasters/shapefiles on a basemap
# the function below is run using an existing folium map. If the folium map defines is my_map, then
# my_map.add_ee_layer(ee_object,name)
# where ee_object is the object defined in google earth engine, and name is the label in folium
def add_ee_layer(self, ee_object, name):
    try:
        # display ee.Image()
        if isinstance(ee_object, ee.image.Image):
            range = ee.Image(ee_object).reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
            vals = range.getInfo()
            min=list(vals.items())[0][1]
            max=list(vals.items())[1][1]
            vis = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}

            map_id_dict = ee.Image(ee_object).getMapId(vis)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
            colormap = cm.LinearColormap(vmin=min,vmax=max,colors=['blue', 'white','red']).to_step(n=10)
            colormap.caption=name
            self.add_child(colormap)
        # display ee.ImageCollection()
        elif isinstance(ee_object, ee.imagecollection.ImageCollection):
            ee_object_new = ee_object.mosaic()
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
        # display ee.Geometry()
        elif isinstance(ee_object, ee.geometry.Geometry):
            folium.GeoJson(
            data = ee_object.getInfo(),
            name = name,
            overlay = True,
            control = True
        ).add_to(self)
        # display ee.FeatureCollection()
        elif isinstance(ee_object, ee.featurecollection.FeatureCollection):
            ee_object_new = ee.Image().paint(ee_object, 0, 2)
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
        ).add_to(self)

    except Exception as e:
        print("Could not display {}".format(name))
        print(e)


# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  min=list(vals.items())[0][1]
  max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
 # range = img.reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
 # vals = range.getInfo()
 # min=list(vals.items())[0][1]
 # max=list(vals.items())[1][1]
 # visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(img)

# load prism data
def get_prism_image(date1,date2,geometry):

  prism = ee.ImageCollection('OREGONSTATE/PRISM/AN81m')
  prism_img = prism.filterDate(date1,date2).select('ppt').mean().clip(geometry)
  return(prism_img) # returns prism average monthly precipitation, in mm

# load landsat 8 data
def get_l8_image(date1,date2,geometry):

  l8 = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT')
  l8_img = l8.filterDate(date1,date2).mean().clip(geometry)
  return(l8_img)

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

# to create an elevation raster from the USGS NED in google earth engine from a user-defined geometry
def get_elev(geometry):

  elev = ee.Image('USGS/NED').clip(geometry)
  return(elev)

# to create an elevation raster from the SRTM in google earth engine from a user-defined geometry
def get_srtm(geometry):

  elev = ee.Image('USGS/SRTMGL1_003').clip(geometry)
  return(elev)

# to create a temporally averaged precipitation raster from GPM from a user-defined geometry
def get_gpm_image(date1,date2,geometry):

  gpm = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07')
  gpm_img = gpm.filterDate(date1,date2).select('precipitation').mean().multiply(24*365/12).clip(geometry) # convert from mm/hour to mm/month
  return(gpm_img) # returns gpm average monthly precipitation in mm

# to create a temporally averaged actual ET raster from the openET ensemble from a user-defined geometry
def get_openET_image(date1,date2,geometry):

  openET = ee.ImageCollection('OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0')
  openET_img = openET.filterDate(date1,date2).select('et_ensemble_mad').mean().clip(geometry)
  return(openET_img)

# to create a temporally averaged reference ET raster from the openET ensemble from a user-defined geometry
def get_RET(date1,date2,geometry):

  ETR = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
  ETR_image = ETR.filterDate(date1,date2).select('etr').mean().multiply(365/12).clip(geometry) # convert from mm/day to mm/month
  return(ETR_image)

# load sentinel 2 data
def get_s2_image(date1,date2,geometry):

    s2 = ee.ImageCollection('COPERNICUS/S2')
    s2_img = s2.filterDate(date1,date2).filterBounds(geometry).first().clip(geometry)
    return(s2_img)

# Add EE drawing method to folium (not a function)
folium.Map.add_ee_layer = add_ee_layer

def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=1000,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):
  """Creates a region reduction function.

  Creates a region reduction function intended to be used as the input function
  to ee.ImageCollection.map() for reducing pixels intersecting a provided region
  to a statistic for each image in a collection. See ee.Image.reduceRegion()
  documentation for more details.

  Args:
    geometry:
      An ee.Geometry that defines the region over which to reduce data.
    reducer:
      Optional; An ee.Reducer that defines the reduction method.
    scale:
      Optional; A number that defines the nominal scale in meters of the
      projection to work in.
    crs:
      Optional; An ee.Projection or EPSG string ('EPSG:5070') that defines
      the projection to work in.
    bestEffort:
      Optional; A Boolean indicator for whether to use a larger scale if the
      geometry contains too many pixels at the given scale for the operation
      to succeed.
    maxPixels:
      Optional; A number specifying the maximum number of pixels to reduce.
    tileScale:
      Optional; A number representing the scaling factor used to reduce
      aggregation tile size; using a larger tileScale (e.g. 2 or 4) may enable
      computations that run out of memory with the default.

  Returns:
    A function that accepts an ee.Image and reduces it by region, according to
    the provided arguments.
  """

  def reduce_region_function(img):
    """Applies the ee.Image.reduceRegion() method.

    Args:
      img:
        An ee.Image to reduce to a statistic by region.

    Returns:
      An ee.Feature that contains properties representing the image region
      reduction results per band and the image timestamp formatted as
      milliseconds from Unix epoch (included to enable time series plotting).
    """

    stat = img.reduceRegion(
        reducer=reducer,
        geometry=geometry,
        scale=scale,
        crs=crs,
        bestEffort=bestEffort,
        maxPixels=maxPixels,
        tileScale=tileScale)

    return ee.Feature(geometry, stat).set({'millis': img.date().millis()})
  return reduce_region_function

# Define a function to transfer feature properties to a dictionary.
def fc_to_dict(fc):
  prop_names = fc.first().propertyNames()
  prop_lists = fc.reduceColumns(
      reducer=ee.Reducer.toList().repeat(prop_names.size()),
      selectors=prop_names).get('list')

  return ee.Dictionary.fromLists(prop_names, prop_lists)

# generate data frame from image collection
def gee_zonal_mean_img_coll(imageCollection,geometry,scale=1000):
    reduce_iC = create_reduce_region_function(geometry = geometry, scale=scale)
    stat_fc = ee.FeatureCollection(imageCollection.map(reduce_iC)).filter(ee.Filter.notNull(imageCollection.first().bandNames()))
    fc_dict = fc_to_dict(stat_fc).getInfo()

    df = pd.DataFrame(fc_dict)
    df['date'] = pd.to_datetime(df['millis'],unit='ms')
    return(df)

def gee_zonal_mean(date1,date2,geometry,collection_name,band_name,scale=1000):
     imcol = ee.ImageCollection(collection_name).select(band_name).filterDate(date1,date2)
     df = gee_zonal_mean_img_coll(imcol,geometry,scale=scale)
     return(df)

# Convert shapefile to EE object
def shapefile_to_ee(filepath):
    # 1. Read the shapefile
    gdf = gpd.read_file(filepath)
    
    # 2. Ensure it is in WGS84 (required by Earth Engine)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    
    # 3. Convert the geometry to a GeoJSON-like mapping
    # This handles Points, Polygons, and MultiPolygons
    geojson = gdf.__geo_interface__
    
    # 4. Create an ee.FeatureCollection from the GeoJSON
    # You can then get the geometry from the collection
    ee_object = ee.FeatureCollection(geojson)
    
    return ee_object.geometry()


# read in shapefile of National Parks
def get_park_system(save: bool = False, path: str | None = None) -> gpd.GeoDataFrame:
    """Import national park boundary shapefile.

    Downloads NPS park boundary shapefiles via the Data Store REST API.

    Reference:
        https://irmaservices.nps.gov/datastore/v6/documentation
        https://irma.nps.gov/DataStore/Reference/Profile/2224545?lnv=True

    Last updated: September 2025

    Args:
        save: If True, write the result to disk as a GeoPackage.
        path: Directory to save the file. Required when save=True.

    Returns:
        GeoDataFrame of all NPS park boundaries (EPSG:4326).
    """
    download_link = "https://irma.nps.gov/DataStore/DownloadFile/733895"

    # ── Download zip to a temp file ────────────────────────────────────────
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        zip_path = tmp_dir / "boundary.zip"
        extract_dir = tmp_dir / "extracted"

        response = requests.get(download_link, stream=True, timeout=120)
        response.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        # ── Unzip and read shapefile ───────────────────────────────────────
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        shp_path = extract_dir / "nps_boundary.shp"
        parks = gpd.read_file(shp_path)

        # ── Fix any invalid geometries (mirrors st_make_valid) ─────────────
        parks["geometry"] = parks["geometry"].make_valid()

        # ── Optionally save as GeoPackage ──────────────────────────────────
        if save and path is not None:
            out_path = (
                Path(path)
                / f"nps_park_boundaries_{date.today()}.gpkg"
            )
            parks.to_file(out_path, driver="GPKG")

    return parks


setup for NPS WBM

In [ ]:
# ── Import the WBM model from the `wbm` package (repo_root/wbm/) ────────────
# All functions are also available via the `wbm` namespace, e.g. wbm.nps_wbm(...)
import importlib
import wbm

# Reload in case you edited the package mid-session without restarting the kernel
importlib.reload(wbm)

from wbm import (
    # ── Raster utilities ──────────────────────────────────────────────────
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run

    # ── Core model ───────────────────────────────────────────────────────
    nps_wbm,                # single-point daily water balance driver
    run_nps_wbm_points,     # multi-point / multi-GCM wrapper

    # ── Component functions (available if needed for custom workflows) ────
    get_freeze,             # rain/snow partitioning factor
    get_rain, get_snow,     # rainfall and snowfall
    get_melt,               # Hock degree-day snowmelt
    get_snowpack,           # snowpack accumulation
    get_ablation,           # snow sublimation / vapor loss
    get_soil,               # soil water content
    get_d_soil,             # daily change in SWC
    get_aet,                # actual evapotranspiration
    get_storage,            # linear storage reservoir (CSU addition)
    get_oudin_pet,          # Oudin PET with topographic heat-load
    get_hamon_pet,          # Hamon PET
    get_penman_monteith_pet,# FAO-56 Penman-Monteith PET
    get_daylength,          # astronomical daylength (hours)
    get_gdd,                # growing degree days
    get_deficit,            # climatic water deficit (PET − AET)

    # ── Low-level helpers (rarely called directly) ────────────────────────
    get_svp, actual_vp, atm_press, psyc_constant,
    vapor_curve, clear_sky_rad, outgoing_rad,
)

print(f"WBM functions loaded from: {_REPO_ROOT / 'wbm'}")


Load raster datasets used for the NPS WBM

In [ ]:

# Reads the three required GeoTIFFs from `Data/wbm_rasters/`, derives slope
# and aspect from the DEM (Horn's 8-neighbour method, matching
# `terra::terrain(neighbors = 8)`), and caches everything in memory.
#
# **Only needs to run once per session.**  After this, `extract_point_params()`
# works without any path arguments.
#
# | File | Content | Units |
# |---|---|---|
# | `elevation_cropped.tif` | DEM | metres |
# | `water_storage.tif` | Soil water storage capacity | cm (auto-converted → mm) |
# | `merged_jennings2.tif` | Jennings temperature climatology | °C |

# %%
# ── Path to raster directory ──────────────────────────────────────────────────
# Adjust if your rasters live elsewhere.
RASTER_DIR = "../Data/wbm_rasters"

# ── Target CRS ────────────────────────────────────────────────────────────────
# Set to None to keep the native CRS of the rasters (EPSG:4326 for NPS data).
# Set to an EPSG string (e.g. "EPSG:26913") to reproject all rasters before
# sampling — useful when your point coordinates are in a projected CRS.
TARGET_CRS = "EPSG:4326"

try:
    load_wbm_rasters(raster_dir=RASTER_DIR, target_crs=TARGET_CRS)
except ImportError as e:
    print(f"⚠️  Rasters not loaded: {e}")
    print("   Install rasterio and re-run this cell before calling extract_point_params().")
except FileNotFoundError as e:
    print(f"⚠️  Raster file not found:\n   {e}")
    print(f"   Check that RASTER_DIR = '{RASTER_DIR}' is correct.")

Define global model settings for NPS WBM

In [ ]:
# Set default WBM run parameters here.  These are passed through to
# `nps_wbm()` / `run_nps_wbm_points()` / `run_pipeline()` in later cells.
# Override any of them at call-time as needed.


# ── PET method ────────────────────────────────────────────────────────────────
# One of: "Oudin"  (default, temperature-based, topographic heat-load adjusted)
#         "Hamon"  (temperature + daylength)
#         "Penman-Monteith"  (requires tmax, tmin, and ideally RH / wind data)
PET_METHOD = "Oudin"

# ── Snowmelt ──────────────────────────────────────────────────────────────────
# Hock (2003) degree-day melt factor (mm °C⁻¹ day⁻¹).
# Hock reports ~2.5 for Gooseberry Creek, UT; NPS default is 4.
HOCK_COEF = 4.0

# ── Initial conditions ────────────────────────────────────────────────────────
SNOWPACK_INIT = 0.0   # mm SWE
SOIL_INIT     = 0.0   # mm

# ── CSU additions ─────────────────────────────────────────────────────────────
DIRECT_FRAC  = 0.0    # fraction of rainfall routed directly to runoff (0–1)
RETURN_RATE  = 1.0    # fraction of storage reservoir released per day (0–1]
PET_MULT     = 1    # multiplicative PET bias correction
SOIL_MULT    = 1    # multiplicative adjustment to SWC_Max

# ── Misc ──────────────────────────────────────────────────────────────────────
SHADE_COEFF  = 1.0    # canopy shading coefficient for Oudin PET (0–1)
T_BASE       = 0.0    # base temperature for growing degree days (°C)
TO_INCHES    = True   # True → output fluxes in inches; False → mm

print("✓ Model settings configured:")
print(f"  PET method   : {PET_METHOD}")
print(f"  Hock coef    : {HOCK_COEF} mm °C⁻¹ day⁻¹")
print(f"  Direct frac  : {DIRECT_FRAC}")
print(f"  Return rate  : {RETURN_RATE}")
print(f"  PET mult     : {PET_MULT}   |  Soil mult: {SOIL_MULT}")
print(f"  Output units : {'inches' if TO_INCHES else 'mm'}")


In [ ]:
# Verify setup (quick sanity check)
#
# Runs the model for one synthetic year at a single point to confirm the full
# stack (imports → raster cache → model) is working before you connect real
# climate data.

# %%
rng = np.random.default_rng(0)
_n  = 365
_dates = pd.date_range("2000-01-01", periods=_n)
_doy   = _dates.dayofyear.to_numpy(float)

_test_climate = pd.DataFrame({
    "date":    _dates,
    "x":       -105.5,          # lon — update to match your study area
    "y":        40.0,           # lat
    "ppt_mm":  rng.exponential(3.0, _n),
    "tmean_C": 8 * np.sin(2 * np.pi * (_doy - 80) / 365) + 5 + rng.normal(0, 2, _n),
    "GCM":     "sanity_check",
})

# Use hard-coded params so the test doesn't depend on the rasters being loaded
_test_params = {"Elev": 2400, "Slope": 10, "Aspect": 180,
                "SWC_Max": 150, "J_Temp": 1.5}

_test_result = nps_wbm(
    _test_climate, _test_params,
    pet_method    = PET_METHOD,
    hock_coef     = HOCK_COEF,
    direct_frac   = DIRECT_FRAC,
    return_rate   = RETURN_RATE,
    pet_mult      = PET_MULT,
    soil_mult     = SOIL_MULT,
    shade_coeff   = SHADE_COEFF,
    t_base        = T_BASE,
    to_inches     = False,          # mm for the sanity check
)

_unit = "mm"
print("✓ Sanity check passed — annual water balance totals:")
print(f"  {'Variable':<14}  {'Annual total':>14}")
print(f"  {'-'*30}")
for _col in ["ppt_mm", "RAIN", "SNOW", "MELT", "AET", "RUNOFF", "D"]:
    print(f"  {_col:<14}  {_test_result[_col].sum():>12.1f} {_unit}")

print(f"\n  Peak snowpack : {_test_result['PACK'].max():.1f} {_unit}")
print(f"  Max soil SWC  : {_test_result['SOIL'].max():.1f} {_unit}")
print(f"\n✓ Setup complete — ready to run NPS WBM.\n")

In [ ]:
# NPS WBM Quick-reference: key function signatures
#
# ```python
# # ── Extract site params at your points from the loaded rasters ────────────
# point_params_df = extract_point_params(points_df)
# # points_df needs columns: x (lon), y (lat)
# # Returns:  Elev, Slope, Aspect, SWC_Max, J_Temp  added to points_df
#
# # ── Run for a single point ────────────────────────────────────────────────
# result = nps_wbm(
#     daily_df     = climate_df,        # date, x, y, ppt_mm, tmean_C [, GCM]
#     point_params = point_params_df.iloc[0].to_dict(),
#     pet_method   = PET_METHOD,
#     **{k: v for k, v in globals().items()
#        if k in ("hock_coef","direct_frac","return_rate","pet_mult",
#                 "soil_mult","shade_coeff","t_base","to_inches")},
# )
#
# # ── Run for multiple points / GCMs ───────────────────────────────────────
# results = run_nps_wbm_points(
#     climate_data    = climate_df,
#     point_params_df = point_params_df,
#     pet_method      = PET_METHOD,
#     aggregate       = True,           # False → keep per-point rows
#     ret_final_cond  = False,          # True → chain into next time period
# )
#
# # ── One-call pipeline (load → extract → run) ─────────────────────────────
# results = run_pipeline(
#     climate_data = climate_df,
#     points_df    = points_df,
#     raster_dir   = RASTER_DIR,
#     pet_method   = PET_METHOD,
# )
# ```

*Loading prerequisites saved by notebooks 01-03 — needed if you're starting this notebook in a fresh kernel. Note: the calibration steps further down in this notebook produce several in-memory results (`OBJECTIVE`, `metrics_*` dataframes, `print_metrics_with_summary`) that notebook 05 currently consumes directly — run notebook 05 in this same kernel session right after this one, without restarting, until those are cached to disk too.*

In [ ]:
# ── Load prerequisites from notebooks 01-03 (fresh kernel = these aren't in memory) ──
flux_towers        = pd.read_csv("../Data/geo_data/flux_towers.csv")
climate_gee         = pd.read_csv("../Data/gridmet_cache/climate_gee_flux_towers_2016_2023.csv",
                                  parse_dates=["date"])
wbm_results         = pd.read_csv("../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
                                  parse_dates=["date"])
wbm_monthly         = pd.read_csv("../Data/gridmet_cache/wbm_monthly_flux_towers_2016_2023.csv",
                                  parse_dates=["date_monthly"])
openet_df           = pd.read_csv("../Data/open_et/openet_timeseries.csv", parse_dates=["date"])
openet_gridcell_df  = pd.read_csv("../Data/open_et_gridcell/openet_gridcell_timeseries.csv",
                                  parse_dates=["date"])

# Site parameters (fast — just samples the already-loaded raster cache)
point_params_df = extract_point_params(flux_towers)

# flux_monthly isn't cached (it's a cheap local read, no GEE call) — rebuild it
flux_frames = []
for row in flux_towers.itertuples():
    fpath = f"../Data/flux_ET_dataset/monthly_data_files/{row.site}_monthly_data.csv"
    if not os.path.exists(fpath):
        continue
    df = pd.read_csv(fpath, parse_dates=["date"])
    df = df[df["date"] >= "2016-01-01"]
    df["site"] = row.site
    flux_frames.append(df)
flux_monthly = (
    pd.concat(flux_frames, ignore_index=True)
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

sites = sorted(flux_towers["site"].unique())

print("Loaded flux_towers, climate_gee, wbm_results, wbm_monthly, openet_df, "
      "openet_gridcell_df, flux_monthly, point_params_df, sites from notebooks 01-03's cache.")


Now we'll start merging and comparing the three datasets:
1. Flux tower ET
2. OpenET ET
3. NPS WBM AET

In [ ]:
# ── Step 1: Align units and column names across all three sources ─────────────

# OpenET ensemble column names -- defined here (not just further down near the
# plotting code) since "Non-null OpenET" below needs ENSEMBLE_COL. Previously
# this only worked because an earlier cell (the OpenET download, back in
# notebook 02) happened to set it first in the same kernel session.
ENSEMBLE_COL = "et_ensemble_mm"
MIN_COL      = "et_ensemble_min_mm"
MAX_COL      = "et_ensemble_max_mm"

MODEL_COLS = [c for c in openet_df.columns
              if c.startswith("et_") and c not in
              [ENSEMBLE_COL, MIN_COL, MAX_COL]]

wbm_plot = wbm_monthly[["site", "date_monthly", "AET"]].copy()
wbm_plot = wbm_plot.rename(columns={"date_monthly": "date"})
if TO_INCHES:
    wbm_plot["AET"] = wbm_plot["AET"] * 25.4
wbm_plot = wbm_plot.rename(columns={"AET": "aet_wbm_mm"})

print("Flux monthly columns:", flux_monthly.columns.tolist())
FLUX_ET_COL = "ET_corr"   

flux_plot = (
    flux_monthly[["site", "date", FLUX_ET_COL]]
    .rename(columns={FLUX_ET_COL: "aet_flux_mm"})
    .assign(date=lambda d: pd.to_datetime(d["date"]))
)

openet_plot = openet_df.copy()
openet_plot["date"] = pd.to_datetime(openet_plot["date"])

def to_month_start(df, date_col="date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.to_period("M").dt.to_timestamp()
    return df

wbm_plot    = to_month_start(wbm_plot)
openet_plot = to_month_start(openet_plot)
flux_plot   = to_month_start(flux_plot)

# ── Gridcell OpenET — merge in ensemble mean only ─────────────────────────────
# ── Gridcell OpenET — merge in ensemble mean, min, and max ───────────────────
openet_gridcell_plot = to_month_start(openet_gridcell_df.copy())
openet_gridcell_plot = openet_gridcell_plot[
    ["site", "date", "et_ensemble_mm", "et_ensemble_min_mm", "et_ensemble_max_mm"]
].rename(columns={
    "et_ensemble_mm":     "et_ensemble_gc_mm",
    "et_ensemble_min_mm": "et_ensemble_gc_min_mm",
    "et_ensemble_max_mm": "et_ensemble_gc_max_mm",
})

# ── Step 2: Merge all sources ─────────────────────────────────────────────────
combined = (
    openet_plot
    .merge(wbm_plot,               on=["site", "date"], how="outer")
    .merge(flux_plot,              on=["site", "date"], how="outer")
    .merge(openet_gridcell_plot,   on=["site", "date"], how="left")   # ← new
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

# Verify overlap
test_site    = sites[0]
wbm_dates    = set(wbm_plot[wbm_plot["site"] == test_site]["date"])
openet_dates = set(openet_plot[openet_plot["site"] == test_site]["date"])
flux_dates   = set(flux_plot[flux_plot["site"] == test_site]["date"])

print("After normalisation:")
print(f"  WBM ∩ OpenET : {len(wbm_dates & openet_dates)} matching months")
print(f"  WBM ∩ Flux   : {len(wbm_dates & flux_dates)} matching months")
print(f"  OpenET ∩ Flux: {len(openet_dates & flux_dates)} matching months")

print(f"\nNon-null WBM        : {combined['aet_wbm_mm'].notna().sum()}")
print(f"Non-null OpenET     : {combined[ENSEMBLE_COL].notna().sum()}")
print(f"Non-null OpenET GC  : {combined['et_ensemble_gc_mm'].notna().sum()}")
print(f"Non-null Flux       : {combined['aet_flux_mm'].notna().sum()}")

# ── Step 3: Plot ──────────────────────────────────────────────────────────────

MODEL_COLORS = {
    "et_disalexi_mm": "#E07B54",
    "et_eemetric_mm": "#5B8DB8",
    "et_geesebal_mm": "#9B59B6",
    "et_ptjpl_mm":    "#E8C84A",
    "et_sims_mm":     "#E8784A",
}
MODEL_LABELS = {
    "et_disalexi_mm": "DisALEXI",
    "et_eemetric_mm": "eeMETRIC",
    "et_geesebal_mm": "geeSEBAL",
    "et_ptjpl_mm":    "PT-JPL",
    "et_sims_mm":     "SIMS",
}

availability = (
    combined.groupby("site")
    .agg(
        n_wbm    = ("aet_wbm_mm",        "count"),
        n_openet = (ENSEMBLE_COL,        "count"),
        n_flux   = ("aet_flux_mm",       "count"),
        n_gc     = ("et_ensemble_gc_mm", "count"),
    )
    .reset_index()
)

print("Data availability per site:")
print(availability.to_string(index=False))

sites_to_plot = availability[
    (availability["n_wbm"]    > 0) &
    (availability["n_openet"] > 0)
]["site"].tolist()

sites_dropped = set(availability["site"]) - set(sites_to_plot)
if sites_dropped:
    print(f"\n⚠️  Dropping {len(sites_dropped)} site(s) with missing WBM or OpenET data:")
    print(f"   {sorted(sites_dropped)}")

print(f"\nSites remaining for plot : {len(sites_to_plot)}")
print(f"Sites with flux data     : "
      f"{availability[availability['n_flux'] > 0]['site'].nunique()}")

combined_plot = combined[combined["site"].isin(sites_to_plot)].copy()

sites   = sorted(sites_to_plot)
n_sites = len(sites)
n_cols  = 4
n_rows  = int(np.ceil(n_sites / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(7, n_rows * 1.2),
    sharey=False,
    constrained_layout=True,
)
axes_flat = axes.flatten()

for ax, site in zip(axes_flat, sites):
    sub = combined_plot[combined_plot["site"] == site].sort_values("date")

    # OpenET ensemble ribbon
    ax.fill_between(
        sub["date"], sub[MIN_COL], sub[MAX_COL],
        color="seagreen", alpha=0.15,
        label="OpenET ensemble min–max",
    )

    # Individual OpenET model lines
    for col in MODEL_COLS:
        if col in sub.columns:
            ax.plot(sub["date"], sub[col],
                    color=MODEL_COLORS.get(col, "grey"),
                    linewidth=0.5, alpha=0.5,
                    label=MODEL_LABELS.get(col, col))

    # Point-sampled OpenET ensemble mean
    ax.plot(sub["date"], sub[ENSEMBLE_COL],
            color="seagreen", linewidth=1.2,
            label="OpenET ensemble (point)", zorder=5)

    # Gridcell-averaged OpenET ensemble mean — darker green   ← new
    ax.plot(sub["date"], sub["et_ensemble_gc_mm"],
            color="#1a5e2a", linewidth=1.5, linestyle="--",
            label="OpenET ensemble (4 km gridcell)", zorder=6)

    # WBM AET
    ax.plot(sub["date"], sub["aet_wbm_mm"],
            color="black", linewidth=1.5,
            linestyle="-", label="WBM AET", zorder=7)

    # Flux tower
    flux_sub = sub.dropna(subset=["aet_flux_mm"])
    if not flux_sub.empty:
        ax.scatter(flux_sub["date"], flux_sub["aet_flux_mm"],
                   color="crimson", s=10, zorder=8,
                   label="Flux tower ET")

    ax.set_title(f"{site}", fontsize=9, fontweight="bold", pad=4)
    ax.set_ylabel("ET (mm / month)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

for ax in axes_flat[n_sites:]:
    ax.set_visible(False)

legend_handles = [
    mpatches.Patch(color="seagreen", alpha=0.3,  label="OpenET ensemble min–max"),
    mlines.Line2D([], [], color="seagreen",  linewidth=1.5,
                  label="OpenET ensemble mean (point)"),
    mlines.Line2D([], [], color="#1a5e2a",   linewidth=1.5, linestyle="--",
                  label="OpenET ensemble mean (4 km gridcell)"),   # ← new
    *[mlines.Line2D([], [], color=MODEL_COLORS[c], linewidth=1,
                    alpha=0.7, label=MODEL_LABELS[c])
      for c in MODEL_COLS if c in MODEL_COLORS],
    mlines.Line2D([], [], color="black", linewidth=1.8,
                  linestyle="-", label="WBM AET"),
    mlines.Line2D([], [], color="crimson", linewidth=0,
                  marker="o", markersize=5, label="Flux tower ET"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=3,
    fontsize=8,
    frameon=False,
    title="Data Sources",
    title_fontsize=9,
)

fig.suptitle(
    "Monthly ET Comparison — Flux Tower Sites Near National Parks",
    fontsize=12, fontweight="bold", y=1.02,
)

plt.tight_layout()
plt.savefig("../Data/open_et/et_comparison_faceted.png", dpi=600, bbox_inches="tight")
plt.show()

# ── Also build combined_gridcell for WBM vs gridcell-averaged OpenET ──────────
combined_gridcell = (
    openet_gridcell_plot
    .merge(wbm_plot,  on=["site", "date"], how="outer")
    .merge(flux_plot, on=["site", "date"], how="outer")
    .rename(columns={"et_ensemble_gc_mm": ENSEMBLE_COL})
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)
print(f"combined_gridcell shape: {combined_gridcell.shape}")




Now, we'll check performance statistics.

In [ ]:

# ── Metric functions ──────────────────────────────────────────────────────────

def slope_through_origin(obs, pred):
    obs, pred = np.asarray(obs), np.asarray(pred)
    return np.dot(obs, pred) / np.dot(obs, obs)

def mbe(obs, pred):
    return np.mean(pred - obs)

def mae(obs, pred):
    return np.mean(np.abs(pred - obs))

def rmse(obs, pred):
    return np.sqrt(np.mean((pred - obs) ** 2))

def r2(obs, pred):
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

def compute_metrics(df, obs_col, pred_col, site_col="site", min_n=6):
    def _slope(obs, pred):
        return np.dot(obs, pred) / np.dot(obs, obs)
    def _mbe(obs, pred):
        return float(np.mean(pred - obs))
    def _mae(obs, pred):
        return float(np.mean(np.abs(pred - obs)))
    def _rmse(obs, pred):
        return float(np.sqrt(np.mean((pred - obs) ** 2)))
    def _r2(obs, pred):
        ss_res = np.sum((obs - pred) ** 2)
        ss_tot = np.sum((obs - np.mean(obs)) ** 2)
        return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    records = []
    for site, grp in df.groupby(site_col):
        paired = grp[[obs_col, pred_col]].dropna()
        n = len(paired)
        if n < min_n:
            records.append({
                "site": site, "n": n,
                "slope": np.nan, "MBE": np.nan,
                "MAE":   np.nan, "RMSE": np.nan, "R2": np.nan,
                "note": f"insufficient data (n={n})",
            })
            continue
        obs  = paired[obs_col].values
        pred = paired[pred_col].values
        records.append({
            "site":  site,
            "n":     n,
            "slope": round(_slope(obs, pred), 3),
            "MBE":   round(_mbe(obs, pred),   3),
            "MAE":   round(_mae(obs, pred),   3),
            "RMSE":  round(_rmse(obs, pred),  3),
            "R2":    round(_r2(obs, pred),    3),
            "note":  "",
        })
    return pd.DataFrame(records).sort_values("site").reset_index(drop=True)


def print_metrics_with_summary(metrics_df, label, metric_cols=None, valid_sites=None):
    """
    Print a per-site metrics table followed by a summary block (mean, median,
    min, max). If valid_sites is provided, summary stats are restricted to
    those sites — ensuring default and optimised comparisons are computed over
    the same set of sites. Sites excluded for insufficient data or not in
    valid_sites are listed but not included in summary stats.
    """
    if metric_cols is None:
        metric_cols = ["slope", "MBE", "MAE", "RMSE", "R2"]

    display_cols = ["site", "n"] + metric_cols + ["note"]
    print(label)
    print(metrics_df[display_cols].to_string(index=False))

    # Filter to valid data, then further restrict to common sites if provided
    valid = metrics_df[metrics_df["note"] == ""]
    if valid_sites is not None:
        valid = valid[valid["site"].isin(valid_sites)]

    n_valid    = len(valid)
    n_excluded = len(metrics_df) - n_valid
    site_note  = f"n={n_valid} sites"
    if valid_sites is not None:
        site_note += ", restricted to common valid sites"

    if n_valid == 0:
        print("  (no sites with sufficient data for summary)\n")
        return

    summary_rows = []
    for stat_label, fn in [("mean",   lambda x: x.mean()),
                            ("median", lambda x: x.median()),
                            ("min",    lambda x: x.min()),
                            ("max",    lambda x: x.max())]:
        row = {"site": f"── {stat_label} ({site_note})", "n": "", "note": ""}
        for col in metric_cols:
            row[col] = round(fn(valid[col]), 3)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)[display_cols]
    sep = "─" * len(metrics_df[display_cols].to_string(index=False).splitlines()[0])
    print(sep)
    print(summary_df.to_string(index=False, header=False))
    if n_excluded > 0:
        excl_sites = metrics_df[~metrics_df["site"].isin(valid["site"])]["site"].tolist()
        print(f"  (excluded from summary: {excl_sites})")
    print()


# ── Build combined_mm (units: mm throughout) ──────────────────────────────────
combined_mm          = combined.copy()
combined_gridcell_mm = combined_gridcell.copy()

# ── Compute baseline per-site metrics ─────────────────────────────────────────
metrics_wbm_vs_openet = compute_metrics(
    combined_gridcell_mm, obs_col=ENSEMBLE_COL, pred_col="aet_wbm_mm")  # ← gridcell
metrics_wbm_vs_openet.insert(0, "comparison", "WBM vs OpenET (4 km gridcell)")

metrics_openet_vs_flux = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col=ENSEMBLE_COL)
metrics_openet_vs_flux.insert(0, "comparison", "OpenET vs Flux")

metrics_wbm_vs_flux = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col="aet_wbm_mm")
metrics_wbm_vs_flux.insert(0, "comparison", "WBM vs Flux")

# ── Determine common valid sites ───────────────────────────────────────────────
common_vs_openet   = set(metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]["site"])
common_vs_flux     = set(metrics_wbm_vs_flux[metrics_wbm_vs_flux["note"] == ""]["site"])
common_openet_flux = set(metrics_openet_vs_flux[metrics_openet_vs_flux["note"] == ""]["site"])

print("── Common valid site sets ────────────────────────────────────────────────")
print(f"  vs OpenET (gridcell) : {sorted(common_vs_openet)}")
print(f"  vs Flux              : {sorted(common_vs_flux)}")
print(f"  OpenET vs Flux       : {sorted(common_openet_flux)}\n")

# ── Print per-site tables with summaries ──────────────────────────────────────
print_metrics_with_summary(metrics_wbm_vs_openet,
    "── WBM AET vs OpenET Ensemble (4 km gridcell) ───────────────────────────",
    valid_sites=common_vs_openet)

print_metrics_with_summary(metrics_openet_vs_flux,
    "── OpenET Ensemble (point) vs Flux Tower ────────────────────────────────",
    valid_sites=common_openet_flux)

print_metrics_with_summary(metrics_wbm_vs_flux,
    "── WBM AET vs Flux Tower ────────────────────────────────────────────────",
    valid_sites=common_vs_flux)

# ── Cross-site aggregate metrics ──────────────────────────────────────────────
comparison_site_map = {
    "WBM vs OpenET (4 km gridcell)": common_vs_openet,
    "OpenET vs Flux":                common_openet_flux,
    "WBM vs Flux":                   common_vs_flux,
}

metrics_baseline = pd.concat(
    [metrics_wbm_vs_openet, metrics_openet_vs_flux, metrics_wbm_vs_flux],
    ignore_index=True,
)

metrics_baseline_filtered = metrics_baseline[
    metrics_baseline.apply(
        lambda r: r["site"] in comparison_site_map.get(r["comparison"], set()),
        axis=1,
    )
]

print("── Cross-site aggregate metrics (baseline, common valid sites) ──────────")
agg_baseline = (
    metrics_baseline_filtered[metrics_baseline_filtered["note"] == ""]
    .groupby("comparison")
    .agg(
        n_sites    = ("site",  "count"),
        slope_mean = ("slope", "mean"),
        MBE_mean   = ("MBE",   "mean"),
        MAE_mean   = ("MAE",   "mean"),
        RMSE_mean  = ("RMSE",  "mean"),
        R2_mean    = ("R2",    "mean"),
    )
    .round(3)
    .reset_index()
)
print(agg_baseline.to_string(index=False))
print("\n(CSV will be saved at the end of the notebook after all bias corrections.)")

In [ ]:

def make_scatter_facets(metrics_df, combined_df, obs_col, pred_col,
                         obs_label, pred_label, title, save_path,
                         n_cols=4):
    """
    Faceted 1:1 scatter plot with regression line and metrics per site.

    Parameters
    ----------
    metrics_df  : output of compute_metrics() for this comparison
    combined_df : combined DataFrame with all ET columns in mm
    obs_col     : column name for reference/observed variable
    pred_col    : column name for modelled/predicted variable
    obs_label   : x-axis label
    pred_label  : y-axis label
    title       : figure suptitle
    save_path   : file path to save figure
    n_cols      : number of columns in facet grid
    """
    # Only plot sites with enough data
    valid_sites = (
        metrics_df[metrics_df["note"] == ""]["site"].tolist()
    )

    if not valid_sites:
        print(f"⚠️  No sites with sufficient paired data for: {title}")
        return

    n_sites = len(valid_sites)
    n_rows  = int(np.ceil(n_sites / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(7, n_rows * 2),
        constrained_layout=True,
    )
    axes_flat = axes.flatten() if n_sites > 1 else [axes]

    for ax, site in zip(axes_flat, valid_sites):
        sub    = combined_df[combined_df["site"] == site].dropna(
                     subset=[obs_col, pred_col])
        obs    = sub[obs_col].values
        pred   = sub[pred_col].values

        # Retrieve pre-computed metrics for annotation
        m = metrics_df[metrics_df["site"] == site].iloc[0]

        # ── Axis limits: shared square range with a little padding ───────────
        all_vals = np.concatenate([obs, pred])
        lo = max(0, np.nanmin(all_vals) * 0.9)
        hi = np.nanmax(all_vals) * 1.1
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        # ── 1:1 line ─────────────────────────────────────────────────────────
        ax.plot([lo, hi], [lo, hi],
                color="grey", linewidth=1, linestyle="--",
                zorder=1, label="1:1")

        # ── OLS regression line (free intercept, for visual reference) ────────
        if len(obs) >= 3:
            slope_ols, intercept, *_ = stats.linregress(obs, pred)
            x_line = np.array([lo, hi])
            ax.plot(x_line, intercept + slope_ols * x_line,
                    color="tomato", linewidth=1.2, zorder=2,
                    label="OLS fit")

        # ── Scatter points ────────────────────────────────────────────────────
        ax.scatter(obs, pred,
                   color="steelblue", s=18, alpha=0.7,
                   zorder=3, edgecolors="none")

        # ── Metric annotation ─────────────────────────────────────────────────
        annotation = (
            f"n = {int(m['n'])}\n"
            f"slope = {m['slope']:.2f}\n"
            f"R² = {m['R2']:.2f}\n"
            f"MBE = {m['MBE']:.1f}\n"
            f"RMSE = {m['RMSE']:.1f}"
        )
        ax.text(0.04, 0.97, annotation,
                transform=ax.transAxes,
                fontsize=6.5, verticalalignment="top",
                family="monospace",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", alpha=0.7, edgecolor="none"))

        ax.set_title(site, fontsize=9, fontweight="bold", pad=4)
        ax.set_xlabel(obs_label,  fontsize=7)
        ax.set_ylabel(pred_label, fontsize=7)
        ax.tick_params(axis="both", labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_aspect("equal", adjustable="box")

    # ── Hide unused axes ──────────────────────────────────────────────────────
    for ax in axes_flat[n_sites:]:
        ax.set_visible(False)

    # ── Shared legend in first empty panel ────────────────────────────────────
    if n_sites < len(axes_flat):
        legend_ax = axes_flat[n_sites]
        legend_ax.set_visible(True)
        legend_ax.axis("off")
        legend_ax.legend(
            handles=[
                mlines.Line2D([], [], color="grey",   linestyle="--",
                              linewidth=1,   label="1:1 line"),
                mlines.Line2D([], [], color="tomato", linestyle="-",
                              linewidth=1.2, label="OLS fit"),
                mlines.Line2D([], [], color="steelblue", linestyle="none",
                              marker="o", markersize=5, label="Monthly ET"),
            ],
            loc="center", fontsize=8, frameon=False,
            title="Legend", title_fontsize=9,
        )

    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")


# ── Convert WBM to mm if needed (use combined_mm from metrics step) ───────────
unit = "inches" if TO_INCHES else "mm"
print(f"WBM units: {unit} — using combined_mm (already converted to mm)")

# ── Plot 1: WBM AET vs gridcell-averaged OpenET ensemble ─────────────────────
make_scatter_facets(
    metrics_df  = metrics_wbm_vs_openet,
    combined_df = combined_gridcell,                        # ← gridcell
    obs_col     = ENSEMBLE_COL,
    pred_col    = "aet_wbm_mm",
    obs_label   = "OpenET Ensemble — grid (mm/month)",   # ← updated
    pred_label  = "WBM AET (mm/month)",
    title       = "WBM AET vs OpenET Ensemble (4 km gridcell)",   # ← updated
    save_path   = "../Data/open_et/scatter_wbm_vs_openet.png",
)

# ── Plot 2: OpenET ensemble vs Flux tower ─────────────────────────────────────
make_scatter_facets(
    metrics_df  = metrics_openet_vs_flux,
    combined_df = combined_mm,
    obs_col     = "aet_flux_mm",
    pred_col    = ENSEMBLE_COL,
    obs_label   = "Flux Tower ET (mm/month)",
    pred_label  = "OpenET Ensemble (mm/month)",
    title       = "OpenET Ensemble vs Flux Tower ET",
    save_path   = "../Data/open_et/scatter_openet_vs_flux.png",
)

# ── Plot 3: WBM AET vs Flux tower ─────────────────────────────────────────────
make_scatter_facets(
    metrics_df  = metrics_wbm_vs_flux,
    combined_df = combined_mm,
    obs_col     = "aet_flux_mm",
    pred_col    = "aet_wbm_mm",
    obs_label   = "Flux Tower ET (mm/month)",
    pred_label  = "WBM AET (mm/month)",
    title       = "WBM AET vs Flux Tower ET",
    save_path   = "../Data/open_et/scatter_wbm_vs_flux.png",
)

In [ ]:
# ── Step 1: Merge metrics with tower coordinates ──────────────────────────────
map_df = (
    metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]
    .merge(flux_towers[["site", "x", "y", "ecosystem"]], on="site", how="left")
    .dropna(subset=["x", "y", "MBE", "RMSE"])
    .reset_index(drop=True)
)

print(f"Sites with metrics + coordinates: {len(map_df)}")
print(map_df[["site", "x", "y", "ecosystem", "MBE", "RMSE",
              "R2", "slope"]].to_string(index=False))

# ── Load US state boundaries (GeoPandas 1.0+ compatible) ─────────────────────

# Option A — download from Census TIGER (requires internet, ~2 MB)
try:
    states = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2020/shp/"
        "cb_2020_us_state_20m.zip"
    )
    print("✓ State boundaries loaded from Census TIGER")

# Option B — geodatasets package (pip install geodatasets)
except Exception:
    try:
        import geodatasets
        states = gpd.read_file(geodatasets.get_path("naturalearth.land"))
        print("✓ State boundaries loaded from geodatasets")

    # Option C — download naturalearth directly from GitHub
    except Exception:
        states = gpd.read_file(
            "https://raw.githubusercontent.com/nvkelso/"
            "natural-earth-vector/master/geojson/"
            "ne_110m_admin_1_states_provinces.geojson"
        )
        print("✓ State boundaries loaded from Natural Earth GitHub")

# Filter to CONUS only (drop Alaska, Hawaii, territories)
NON_CONUS = ["Alaska", "Hawaii", "Puerto Rico", "Guam",
             "United States Virgin Islands", "American Samoa",
             "Commonwealth of the Northern Mariana Islands"]

# Column name differs by source — handle both
name_col = "NAME" if "NAME" in states.columns else "name"
states_conus = states[~states[name_col].isin(NON_CONUS)].copy()

print(f"  States in plot: {len(states_conus)}")

# Fallback if no internet: use naturalearth countries clipped to CONUS
# states = world[world["continent"] == "North America"]

# CONUS bounding box
LON_MIN, LON_MAX = -125, -65
LAT_MIN, LAT_MAX =   24,  50

# ── Step 3: Colormap helpers ───────────────────────────────────────────────────
# MBE: diverging — blue = underestimate, red = overestimate
mbe_abs_max = np.ceil(map_df["MBE"].abs().max() / 10) * 10
mbe_norm    = mcolors.TwoSlopeNorm(vmin=-mbe_abs_max, vcenter=0, vmax=mbe_abs_max)
mbe_cmap    = "RdBu_r"

# RMSE: sequential — light to dark
rmse_norm = mcolors.Normalize(vmin=0, vmax=np.ceil(map_df["RMSE"].max() / 10) * 10)
rmse_cmap = "YlOrRd"

# ── Step 4: Two-panel map figure ──────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6),
    constrained_layout=True,
)

for ax, metric, cmap, norm, label in [
    (axes[0], "MBE",  mbe_cmap,  mbe_norm,  "MBE (mm/month)\nnegative = underestimate"),
    (axes[1], "RMSE", rmse_cmap, rmse_norm, "RMSE (mm/month)"),
]:
    # ── Basemap ───────────────────────────────────────────────────────────────
    states.plot(ax=ax, color="whitesmoke", edgecolor="grey",
                linewidth=0.4, zorder=1)

    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect("equal")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude",  fontsize=8)

    # ── Scatter points sized by |value|, coloured by value ───────────────────
    vals    = map_df[metric].values
    sizes   = (np.abs(vals) / np.abs(vals).max() * 300).clip(30)

    sc = ax.scatter(
        map_df["x"], map_df["y"],
        c       = vals,
        s       = sizes,
        cmap    = cmap,
        norm    = norm,
        zorder  = 5,
        edgecolors = "k",
        linewidths = 0.5,
        alpha   = 0.9,
    )

    # ── Site labels ───────────────────────────────────────────────────────────
    for _, row in map_df.iterrows():
        ax.annotate(
            f"{row['site']}\n{row[metric]:.1f}",
            xy       = (row["x"], row["y"]),
            xytext   = (6, 4),
            textcoords = "offset points",
            fontsize = 6,
            color    = "black",
            zorder   = 6,
        )

    # ── Colorbar ──────────────────────────────────────────────────────────────
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02, aspect=20)
    cbar.set_label(label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    # ── Size legend ───────────────────────────────────────────────────────────
    if metric == "RMSE":
        # Show representative sizes
        ref_vals  = [50, 150, 300]
        ref_sizes = [v / np.abs(vals).max() * 300 for v in ref_vals]
        size_handles = [
            plt.scatter([], [], s=s, color="grey",
                        edgecolors="k", linewidths=0.5,
                        alpha=0.7, label=f"{v:.0f} mm")
            for s, v in zip(ref_sizes, ref_vals)
            if v <= np.abs(vals).max()
        ]
        ax.legend(
            handles=size_handles,
            title="RMSE magnitude",
            title_fontsize=7,
            fontsize=7,
            loc="lower left",
            frameon=True,
            framealpha=0.8,
        )
    else:
        ref_vals  = [-100, -50, 50, 100]
        ref_sizes = [abs(v) / np.abs(vals).max() * 300 for v in ref_vals]
        size_handles = [
            plt.scatter([], [], s=s, color="grey",
                        edgecolors="k", linewidths=0.5,
                        alpha=0.7, label=f"{v:+.0f} mm")
            for s, v in zip(ref_sizes, ref_vals)
            if abs(v) <= np.abs(vals).max()
        ]
        ax.legend(
            handles=size_handles,
            title="MBE magnitude",
            title_fontsize=7,
            fontsize=7,
            loc="lower left",
            frameon=True,
            framealpha=0.8,
        )

    ax.set_title(
        f"WBM AET vs OpenET (4 km gridcell) — {metric}",
        fontsize=11, fontweight="bold", pad=8,
    )

fig.suptitle(
    "WBM AET vs OpenET Ensemble (4 km gridcell): spatial distribution of bias and error\n"
    "Point size proportional to magnitude",
    fontsize=12, fontweight="bold",
)

plt.savefig("../Data/open_et/map_wbm_vs_openet_bias_rmse.png",
            dpi=150, bbox_inches="tight")
plt.show()

Now, we'll make a few attempts to correct the WBM AET based on OpenET. 

First, we'll apply a correction factor directly to the WBM AET output to match OpenET. 

In [ ]:
# ── Step 1: Align units and column names ──────────────────────────────────────

# WBM: inches → mm
wbm_plot = wbm_monthly[["site", "date_monthly", "AET", "ecosystem", "state"]].copy()
wbm_plot = wbm_plot.rename(columns={"date_monthly": "date"})
if TO_INCHES:
    wbm_plot["AET"] = wbm_plot["AET"] * 25.4   # unit conversion only
wbm_plot = wbm_plot.rename(columns={"AET": "aet_wbm_mm"})

# Flux tower
flux_plot = (
    flux_monthly[["site", "date", "ET_corr"]]
    .rename(columns={"ET_corr": "aet_flux_mm"})
    .assign(date=lambda d: pd.to_datetime(d["date"]))
)

# OpenET (point-sampled — for flux tower comparisons)
openet_plot = openet_df.copy()
openet_plot["date"] = pd.to_datetime(openet_plot["date"])

# Gridcell-averaged OpenET (for WBM vs OpenET comparisons)
openet_gridcell_plot_cal = openet_gridcell_df.copy()
openet_gridcell_plot_cal["date"] = pd.to_datetime(openet_gridcell_plot_cal["date"])

# Normalise all dates to first of month
def to_month_start(df, date_col="date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.to_period("M").dt.to_timestamp()
    return df

wbm_plot    = to_month_start(wbm_plot)
openet_plot = to_month_start(openet_plot)
flux_plot   = to_month_start(flux_plot)

# ── Step 2: Merge ─────────────────────────────────────────────────────────────
combined_mm = (
    openet_plot
    .merge(wbm_plot,  on=["site", "date"], how="outer")
    .merge(flux_plot, on=["site", "date"], how="outer")
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

# Gridcell-averaged OpenET merged with WBM and flux — for WBM vs OpenET
combined_gridcell_mm = (
    to_month_start(openet_gridcell_plot_cal)
    .merge(to_month_start(wbm_plot),  on=["site", "date"], how="outer")
    .merge(to_month_start(flux_plot), on=["site", "date"], how="outer")
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

# ── Step 3: Compute OLS correction per site ───────────────────────────────────
# OLS with intercept: corrected = (WBM - intercept) / slope
# This better handles high-ET underestimation vs. slope-through-origin

metrics_wbm_vs_openet = compute_metrics(
    combined_gridcell_mm,
    obs_col  = ENSEMBLE_COL,
    pred_col = "aet_wbm_mm",
)

ols_params = {}
# Use gridcell OpenET to calibrate WBM
for site, grp in combined_gridcell_mm.groupby("site"):
    paired = grp[[ENSEMBLE_COL, "aet_wbm_mm"]].dropna()
    if len(paired) >= 6:
        slope, intercept, r, _, _ = stats.linregress(
            paired[ENSEMBLE_COL], paired["aet_wbm_mm"]
        )
        r2_val = r ** 2
        ols_params[site] = {
            "ols_slope":     slope,
            "ols_intercept": intercept,
            "R2":            r2_val,
        }

ols_df = (
    pd.DataFrame(ols_params)
    .T.reset_index()
    .rename(columns={"index": "site"})
    .astype({"ols_slope": float, "ols_intercept": float, "R2": float})
)

# Minimum R² required to apply calibration; sites below this threshold
# retain a multiplier of 1.0 (no correction applied).
R2_THRESHOLD = 0.5

ols_df["calibration_applied"] = ols_df["R2"] > R2_THRESHOLD

print("OLS correction parameters per site:")
print(ols_df.to_string(index=False))

# ── Derive et_multipliers from ols_df ────────────────────────────────────────
# The multiplier is 1/ols_slope (inverse of the regression slope), clipped to a
# safe range to prevent extreme corrections. Sites below R2_THRESHOLD keep
# et_multiplier_safe = 1.0 (no change applied).
MAX_SAFE_MULT = 3.0
et_multipliers = ols_df.copy()
et_multipliers["et_multiplier"] = np.where(
    et_multipliers["ols_slope"].abs() > 0.01,
    1.0 / et_multipliers["ols_slope"],
    1.0,
)
et_multipliers["et_multiplier_safe"] = (
    et_multipliers["et_multiplier"]
    .clip(1.0 / MAX_SAFE_MULT, MAX_SAFE_MULT)
)
# Force multiplier to 1.0 for sites that do not meet the R² threshold
et_multipliers.loc[
    ~et_multipliers["calibration_applied"], "et_multiplier_safe"
] = 1.0

print("OLS-derived multipliers per site:")
print(et_multipliers[["site", "ols_slope", "ols_intercept", "R2",
                       "calibration_applied", "et_multiplier_safe"]].to_string(index=False))

# ── Step 4: Apply calibration ─────────────────────────────────────────────────
combined_mm = combined_mm.merge(
    et_multipliers[["site", "et_multiplier_safe", "calibration_applied"]],
    on="site", how="left"
)
combined_mm["aet_wbm_cal_mm"] = (
    combined_mm["aet_wbm_mm"] * combined_mm["et_multiplier_safe"]
)

# Apply same calibration multipliers to gridcell-merged data
combined_gridcell_mm = combined_gridcell_mm.merge(
    et_multipliers[["site", "et_multiplier_safe", "calibration_applied"]],
    on="site", how="left"
)
combined_gridcell_mm["aet_wbm_cal_mm"] = (
    combined_gridcell_mm["aet_wbm_mm"] * combined_gridcell_mm["et_multiplier_safe"]
)

# Define column groups for plotting
ENSEMBLE_COL = "et_ensemble_mm"
MIN_COL      = "et_ensemble_min_mm"
MAX_COL      = "et_ensemble_max_mm"
MODEL_COLS   = [c for c in openet_df.columns
                if c.startswith("et_") and c not in
                [ENSEMBLE_COL, MIN_COL, MAX_COL]]
MODEL_COLORS = {
    "et_disalexi_mm": "#E07B54", "et_eemetric_mm": "#5B8DB8",
    "et_geesebal_mm": "#9B59B6", "et_ptjpl_mm":    "#E8C84A",
    "et_sims_mm":     "#E8784A",
}
MODEL_LABELS = {
    "et_disalexi_mm": "DisALEXI", "et_eemetric_mm": "eeMETRIC",
    "et_geesebal_mm": "geeSEBAL", "et_ptjpl_mm":    "PT-JPL",
    "et_sims_mm":     "SIMS",
}

# Sanity check
print("\nSanity check — calibrated should be >= uncalibrated:")
check = (
    combined_mm.groupby("site")
    .agg(
        multiplier  = ("et_multiplier_safe", "first"),
        cal_applied = ("calibration_applied", "first"),
        wbm_mean    = ("aet_wbm_mm",          "mean"),
        cal_mean    = ("aet_wbm_cal_mm",       "mean"),
        openet_mean = (ENSEMBLE_COL,           "mean"),
    )
    .reset_index()
)
#check["cal_closer"] = (
#    abs(check["cal_mean"] - check["openet_mean"]) 
#    abs(check["wbm_mean"] - check["openet_mean"])
#)
check["cal_closer"] = abs(check["cal_mean"] - check["openet_mean"]) < abs(check["wbm_mean"] - check["openet_mean"])
print(check.to_string(index=False))

# ── Step 5: Recompute metrics ─────────────────────────────────────────────────
metrics_cal_vs_openet = compute_metrics(
    combined_gridcell_mm, obs_col=ENSEMBLE_COL, pred_col="aet_wbm_cal_mm"
)
metrics_cal_vs_flux = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col="aet_wbm_cal_mm"
)

print("\n── Before vs after calibration ──────────────────────────────────────────")
for label, mdf in [("Uncalibrated", metrics_wbm_vs_openet),
                    ("Calibrated",   metrics_cal_vs_openet)]:
    valid = mdf[mdf["note"] == ""]
    print(f"\n  {label}:")
    for metric in ["slope", "MBE", "MAE", "RMSE", "R2"]:
        print(f"    {metric:<5} mean = {valid[metric].mean():>8.3f}")

# ── Step 6: Filter to plottable sites ─────────────────────────────────────────
availability = (
    combined_mm.groupby("site")
    .agg(n_wbm=("aet_wbm_mm", "count"),
         n_openet=(ENSEMBLE_COL, "count"),
         n_flux=("aet_flux_mm", "count"))
    .reset_index()
)
sites_to_plot = availability[
    (availability["n_wbm"] > 0) & (availability["n_openet"] > 0)
]["site"].tolist()

sites_dropped = set(availability["site"]) - set(sites_to_plot)
if sites_dropped:
    print(f"\n⚠️  Dropping sites with no WBM or OpenET data: {sorted(sites_dropped)}")

combined_plot = combined_mm[combined_mm["site"].isin(sites_to_plot)].copy()

# ── Merge gridcell ensemble into combined_plot ────────────────────────────────
gc_mean = (
    combined_gridcell_mm[["site", "date", ENSEMBLE_COL]]
    .rename(columns={ENSEMBLE_COL: "et_ensemble_gc_mm"})
    .dropna(subset=["et_ensemble_gc_mm"])
)
combined_plot = combined_plot.merge(gc_mean, on=["site", "date"], how="left")

sites   = sorted(sites_to_plot)
n_sites = len(sites)
n_cols  = 4
n_rows  = int(np.ceil(n_sites / n_cols))

# ── Step 7: Faceted time series plot ──────────────────────────────────────────
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(7, n_rows * 1.5),
    sharey=False, constrained_layout=True,
)
axes_flat = axes.flatten()

for ax, site in zip(axes_flat, sites):
    sub  = combined_plot[combined_plot["site"] == site].sort_values("date")
    m    = metrics_cal_vs_openet[metrics_cal_vs_openet["site"] == site]
    cal  = sub["calibration_applied"].iloc[0] if "calibration_applied" in sub else False
    mult = sub["et_multiplier_safe"].iloc[0]  if "et_multiplier_safe"  in sub else 1.0

    # OpenET point ensemble ribbon
    #ax.fill_between(sub["date"], sub[MIN_COL], sub[MAX_COL],
    #                color="seagreen", alpha=0.15,
    #                label="OpenET ensemble min–max (point)")

    # Point-sampled OpenET ensemble mean
    #ax.plot(sub["date"], sub[ENSEMBLE_COL],
    #        color="seagreen", linewidth=1.5, zorder=5,
    #        label="OpenET ensemble (point)")

    # Gridcell-averaged OpenET ensemble mean
    ax.plot(sub["date"], sub["et_ensemble_gc_mm"],
            color="#1a5e2a", linewidth=1.5, linestyle="-", zorder=6,
            label="OpenET ensemble (4 km gridcell)")

    # Uncalibrated WBM
    ax.plot(sub["date"], sub["aet_wbm_mm"],
            color="black", linewidth=1,
            linestyle=":", alpha=0.8, zorder=4,
            label="WBM AET (uncal.)")

    # Calibrated WBM
    ax.plot(sub["date"], sub["aet_wbm_cal_mm"],
            color="black", linewidth=1.8,
            linestyle="-", zorder=7,
            label="WBM AET (cal.)")

    # Flux tower scatter
    flux_sub = sub.dropna(subset=["aet_flux_mm"])
    if not flux_sub.empty:
        ax.scatter(flux_sub["date"], flux_sub["aet_flux_mm"],
                   color="crimson", s=10, zorder=8,
                   label="Flux tower ET")

    # Annotation
    r2v     = m["R2"].iloc[0] if not m.empty else np.nan
    cal_str = f"k={mult:.2f}" if cal else "no cal."
    ax.annotate(f"{cal_str}  R²={r2v:.2f}",
                xy=(0.04, 0.93), xycoords="axes fraction",
                fontsize=6.5,
                color="steelblue" if cal else "grey",
                bbox=dict(boxstyle="round,pad=0.2",
                          facecolor="white", alpha=0.7,
                          edgecolor="none"))

    ax.set_title(f"{site}", fontsize=9, fontweight="bold", pad=4)
    ax.set_ylabel("ET (mm / month)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

for ax in axes_flat[n_sites:]:
    ax.set_visible(False)

if n_sites < len(axes_flat):
    legend_ax = axes_flat[n_sites]
    legend_ax.set_visible(True)
    legend_ax.axis("off")
    legend_ax.legend(
        handles=[
            #mpatches.Patch(color="seagreen", alpha=0.3,
            #               label="OpenET ensemble min–max (point)"),
            #mlines.Line2D([], [], color="seagreen", linewidth=1.5,
            #              label="OpenET ensemble (point)"),
            mlines.Line2D([], [], color="#1a5e2a", linewidth=1.5,
                          linestyle="-", label="OpenET ensemble (4 km gridcell)"),
            mlines.Line2D([], [], color="black", linewidth=1,
                          linestyle=":", alpha=0.8,
                          label="WBM AET (uncalibrated)"),
            mlines.Line2D([], [], color="black", linewidth=1.8,
                          linestyle="-", label="WBM AET (calibrated)"),
            mlines.Line2D([], [], color="crimson", linewidth=0,
                          marker="o", markersize=5, label="Flux tower ET"),
        ],
        loc="center", fontsize=8, frameon=False,
        title="Data sources", title_fontsize=9,
    )

fig.suptitle(
    "Monthly ET comparison — NPS WBM before/after calibration vs OpenET\n"
    f"Calibration where R² > {R2_THRESHOLD}  |  k = 1/slope (scales WBM up)",
    fontsize=12, fontweight="bold",
)
plt.savefig("../Data/open_et/et_comparison_calibrated.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Step 8: Scatter plots ─────────────────────────────────────────────────────
make_scatter_facets(
    metrics_df  = metrics_cal_vs_openet,
    combined_df = combined_gridcell_mm,
    obs_col     = ENSEMBLE_COL,
    pred_col    = "aet_wbm_cal_mm",
    obs_label   = "OpenET Ensemble — 4 km gridcell (mm/month)",
    pred_label  = "WBM AET calibrated (mm/month)",
    title       = "Calibrated WBM AET vs OpenET Ensemble (4 km gridcell)",
    save_path   = "../Data/open_et/scatter_wbm_cal_vs_openet.png",
)
make_scatter_facets(
    metrics_df  = metrics_cal_vs_flux,
    combined_df = combined_plot,
    obs_col     = "aet_flux_mm",
    pred_col    = "aet_wbm_cal_mm",
    obs_label   = "Flux Tower ET (mm/month)",
    pred_label  = "WBM AET calibrated (mm/month)",
    title       = "Calibrated WBM AET vs Flux Tower ET",
    save_path   = "../Data/open_et/scatter_wbm_cal_vs_flux.png",
)



The WBM AET results are still bad. So, instead we'll apply a multiplier to pet within the model to improve AET calculation.

In [ ]:
# Defines four reusable functions used by every cell below.
# Run once; no output expected.
# ═══════════════════════════════════════════════════════════════════════════════
 
 
# ── A1: Date normalisation ────────────────────────────────────────────────────
def to_month_start(df, date_col="date"):
    """Normalise any date column to the first day of its month (in-place copy)."""
    df = df.copy()
    df[date_col] = (
        pd.to_datetime(df[date_col])
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    return df
 
 
# ── A2: WBM runner with spin-up ───────────────────────────────────────────────
def run_wbm_with_spinup(site_climate, params_row, pet_mult_val,
                         n_spinup=3, to_inches=None):
    """
    Run the NPS WBM with a warm-start spin-up and return the full daily result.
 
    Spin-up iterates the first calendar year N times so soil moisture
    equilibrates before the main run. Uses all global model settings
    (PET_METHOD, HOCK_COEF, etc.) defined in the setup notebook.
 
    Parameters
    ----------
    site_climate  : DataFrame — daily climate for one site
    params_row    : named tuple row from point_params_df (has Elev, Slope …)
    pet_mult_val  : float — PET multiplier for this run
    n_spinup      : int   — spin-up iterations (default 3)
    to_inches     : bool  — if None, uses the global TO_INCHES setting
 
    Returns
    -------
    pd.DataFrame — daily WBM output (same columns as nps_wbm())
    """
    _to_inches = TO_INCHES if to_inches is None else to_inches
    scale = 25.4 if _to_inches else 1.0
 
    pp = {
        "Elev":    params_row.Elev,
        "Slope":   params_row.Slope,
        "Aspect":  params_row.Aspect,
        "SWC_Max": params_row.SWC_Max,
        "J_Temp":  params_row.J_Temp,
    }
 
    # Spin-up: run the first year repeatedly to equilibrate soil
    sp_init, s_init = 0.0, 0.0
    spinup_climate = site_climate[
        site_climate["date"].dt.year == site_climate["date"].dt.year.min()
    ].copy()
 
    for _ in range(n_spinup):
        r = nps_wbm(
            spinup_climate, pp,
            pet_method=PET_METHOD, hock_coef=HOCK_COEF,
            direct_frac=DIRECT_FRAC, return_rate=RETURN_RATE,
            pet_mult=pet_mult_val, soil_mult=SOIL_MULT,
            shade_coeff=SHADE_COEFF, t_base=T_BASE,
            to_inches=_to_inches,
            snowpack_init=sp_init, soil_init=s_init,
        )
        sp_init = float(r["PACK"].iloc[-1]) * scale
        s_init  = float(r["SOIL"].iloc[-1]) * scale
 
    # Main run with spun-up initial conditions
    return nps_wbm(
        site_climate, pp,
        pet_method=PET_METHOD, hock_coef=HOCK_COEF,
        direct_frac=DIRECT_FRAC, return_rate=RETURN_RATE,
        pet_mult=pet_mult_val, soil_mult=SOIL_MULT,
        shade_coeff=SHADE_COEFF, t_base=T_BASE,
        to_inches=_to_inches,
        snowpack_init=sp_init, soil_init=s_init,
    )
 
 
# ── A3: Monthly aggregator ────────────────────────────────────────────────────
def make_wbm_monthly(wbm_daily, flux_cols=None, state_cols=None):
    """
    Aggregate daily WBM output to monthly, using sum for fluxes and
    mean for state variables. Converts AET to mm regardless of TO_INCHES.
 
    Returns a DataFrame with date_monthly, site, ecosystem, state,
    all aggregated WBM columns, and aet_mm (always in mm).
    """
    if flux_cols is None:
        flux_cols  = ["ppt_mm", "RAIN", "SNOW", "MELT", "AET",
                      "RUNOFF", "D", "etr_gridmet_mm"]
    if state_cols is None:
        state_cols = ["SOIL", "PACK", "tmean_C"]
 
    agg = {c: "sum"  for c in flux_cols  if c in wbm_daily.columns}
    agg.update({c: "mean" for c in state_cols if c in wbm_daily.columns})
 
    monthly = (
        wbm_daily
        .assign(date_monthly=lambda d:
                pd.to_datetime(d["date"].dt.to_period("M").dt.to_timestamp()))
        .groupby(["site", "ecosystem", "state", "date_monthly"], as_index=False)
        .agg(agg)
        .sort_values(["site", "date_monthly"])
        .reset_index(drop=True)
    )
 
    # Always store AET in mm for downstream merging
    scale = 25.4 if TO_INCHES else 1.0
    monthly["aet_mm"] = monthly["AET"] * scale
    return monthly
 
 
# ── A4: Canonical combined_mm builder ─────────────────────────────────────────
def build_combined_mm(openet_df, flux_monthly, wbm_monthly_dict, openet_gridcell_df=None):
    """
    Build combined_mm from scratch by outer-joining all ET sources.
 
    Avoids the incremental left-merge corruption that causes missing data
    in plots. Every source is normalised to month-start dates before merging.
 
    Parameters
    ----------
    openet_df            : point-sampled OpenET DataFrame (flux tower comparison)
    openet_gridcell_df   : gridcell-averaged OpenET DataFrame (WBM comparison).
                           If provided, gridcell ensemble columns are added to
                           the output with the suffix _gc (e.g. et_ensemble_gc_mm).
    flux_monthly     : flux tower monthly DataFrame (has ET_corr column)
    wbm_monthly_dict : dict of {label: wbm_monthly_df}
                       e.g. {"aet_wbm_mm":    wbm_monthly,
                             "aet_petcal_mm": wbm_monthly_petcal,
                             "aet_peak_mm":   wbm_monthly_peak}
                       Each df must have site, date_monthly, aet_mm columns.
 
    Returns
    -------
    pd.DataFrame — one row per (site, month) with all ET columns aligned.
    """
    openet_plot = to_month_start(openet_df.copy())

    flux_plot = (
        flux_monthly[["site", "date", "ET_corr"]]
        .rename(columns={"ET_corr": "aet_flux_mm"})
        .assign(date=lambda d: pd.to_datetime(d["date"]))
        .pipe(to_month_start)
    )

    combined = openet_plot.copy()

    for col_name, monthly_df in wbm_monthly_dict.items():
        m = monthly_df.copy()

        # ── Compute aet_mm on the fly if not already present ──────────────────
        if "aet_mm" not in m.columns:
            if "AET" in m.columns:
                scale = 25.4 if TO_INCHES else 1.0
                m["aet_mm"] = m["AET"] * scale
            else:
                raise KeyError(
                    f"WBM DataFrame for '{col_name}' has neither 'aet_mm' "
                    f"nor 'AET'. Available columns: {m.columns.tolist()}"
                )

        # ── Normalise date column name ─────────────────────────────────────────
        date_col = "date_monthly" if "date_monthly" in m.columns else "date"
        m = m.rename(columns={date_col: "date"})
        m = to_month_start(m)

        # ── Select only what we need for the merge ────────────────────────────
        keep = ["site", "date", "aet_mm"]
        for extra in ["ecosystem", "state"]:
            if extra in m.columns and extra not in combined.columns:
                keep.append(extra)

        combined = combined.merge(
            m[keep].rename(columns={"aet_mm": col_name}),
            on=["site", "date"], how="outer"
        )

    combined = combined.merge(flux_plot, on=["site", "date"], how="outer")

    # ── Optional: merge gridcell-averaged OpenET columns (WBM comparison) ─────
    if openet_gridcell_df is not None:
        gc_plot = to_month_start(openet_gridcell_df.copy())
        gc_rename = {
            "et_ensemble_mm":     "et_ensemble_gc_mm",
            "et_ensemble_min_mm": "et_ensemble_gc_min_mm",
            "et_ensemble_max_mm": "et_ensemble_gc_max_mm",
        }
        # Also rename individual model columns
        for col in gc_plot.columns:
            if col.startswith("et_") and col not in gc_rename:
                gc_rename[col] = col.replace("et_", "et_gc_", 1)
        gc_cols = ["site", "date"] + [c for c in gc_plot.columns
                                       if c.startswith("et_")]
        combined = combined.merge(
            gc_plot[gc_cols].rename(columns=gc_rename),
            on=["site", "date"], how="outer"
        )

    combined = (
        combined
        .sort_values(["site", "date"])
        .reset_index(drop=True)
    )
    return combined
 
 
print("✓ Shared helpers defined: to_month_start, run_wbm_with_spinup, "
      "make_wbm_monthly, build_combined_mm")

In [ ]:
# ── B0: Configuration ─────────────────────────────────────────────────────────
# Choose objective function
# Options: "rmse_standard" | "rmse_weighted" | "percentile_bias" | "peak_nse"
OBJECTIVE = "peak_nse"
 
# Preview site — show a before/after plot for this site before running all
TEST_SITE = "US-Ro1"
 
# Search range and resolution
COARSE_LO   = 0.5
COARSE_HI   = 8.0
COARSE_STEP = 0.5
MAX_MULT    = 10.0      # hard cap on PET_mult

# ── Calibration / held-out test split ─────────────────────────────────────────
# Fit PET_mult using ONLY the calibration period; the WBM itself still runs the
# full 2016-2023 record for spin-up/continuity, but the objective function
# below only ever sees calibration-period matched months. This keeps the
# 2022-2023 test period genuinely held out, so notebook 07 can report honest,
# out-of-sample skill for the calibrated variants instead of in-sample fit
# stats.
CAL_START  = "2016-01-01"
CAL_END    = "2021-12-31"
TEST_START = "2022-01-01"
TEST_END   = "2023-12-31"


# ── Gridcell OpenET column names ──────────────────────────────────────────────
GRIDCELL_ENSEMBLE_COL = "et_ensemble_gc_mm"
GRIDCELL_MIN_COL      = "et_ensemble_gc_min_mm"
GRIDCELL_MAX_COL      = "et_ensemble_gc_max_mm"


# ── B1: Objective function definitions ────────────────────────────────────────
def _rmse_standard(obs, pred):
    return float(np.sqrt(np.mean((obs - pred) ** 2)))
 
def _rmse_weighted(obs, pred):
    w = obs / obs.sum()
    return float(np.sqrt(np.sum(w * (obs - pred) ** 2)))
 
def _percentile_bias(obs, pred, pct=90):
    return float(abs(np.percentile(obs, pct) - np.percentile(pred, pct)))
 
def _peak_nse(obs, pred, top_frac=0.25):
    threshold  = np.quantile(obs, 1 - top_frac)
    mask       = obs >= threshold
    obs_p, pred_p = obs[mask], pred[mask]
    ss_res = np.sum((obs_p - pred_p) ** 2)
    ss_tot = np.sum((obs_p - np.mean(obs_p)) ** 2)
    nse    = 1 - ss_res / ss_tot if ss_tot > 0 else -9999
    return 1 - nse   # minimise → maximise NSE
 
OBJECTIVE_FNS = {
    "rmse_standard":   _rmse_standard,
    "rmse_weighted":   _rmse_weighted,
    "percentile_bias": _percentile_bias,
    "peak_nse":        _peak_nse,
}
 
 
# ── B2: Single-site objective wrapper ─────────────────────────────────────────
def make_site_objective(site_climate, site_openet, params_row, obj_name):
    """
    Returns a scalar function of pet_mult_val for one site.
    site_openet must have columns: date, ENSEMBLE_COL
    """
    obj_fn = OBJECTIVE_FNS[obj_name]
 
    def _obj(pet_mult_val):
        try:
            daily   = run_wbm_with_spinup(site_climate, params_row, pet_mult_val)
            scale   = 25.4 if TO_INCHES else 1.0
            monthly = (
                daily
                .assign(
                    date_m = lambda d:
                        pd.to_datetime(d["date"].dt.to_period("M").dt.to_timestamp()),
                    AET_mm = lambda d: d["AET"] * scale,
                )
                .groupby("date_m", as_index=False)["AET_mm"].sum()
                .rename(columns={"date_m": "date"})
            )
            merged = site_openet.merge(monthly, on="date", how="inner")
            if len(merged) < 6:
                return 9999.0
            return obj_fn(merged[ENSEMBLE_COL].values, merged["AET_mm"].values)
        except Exception:
            return 9999.0
 
    return _obj
 
 
# ── B3: Single-site preview ───────────────────────────────────────────────────
print(f"── Single-site preview: {TEST_SITE} ─────────────────────────────────")
 
test_climate = climate_gee[climate_gee["site"] == TEST_SITE].copy()
test_params  = point_params_df[point_params_df["site"] == TEST_SITE].iloc[0]
# Use gridcell-averaged OpenET for the WBM optimization objective
test_openet = (
    combined[combined["site"] == TEST_SITE]
    [["date", GRIDCELL_ENSEMBLE_COL, GRIDCELL_MIN_COL, GRIDCELL_MAX_COL, "aet_flux_mm"]]
    .dropna(subset=[GRIDCELL_ENSEMBLE_COL])
    .rename(columns={
        GRIDCELL_ENSEMBLE_COL: ENSEMBLE_COL,
        GRIDCELL_MIN_COL:      MIN_COL,
        GRIDCELL_MAX_COL:      MAX_COL,
    })
)
 
# Fit only on the calibration period -- test_openet itself stays full-period
# (used below for the before/after plot), so you can see how the calibrated
# multiplier generalises into the held-out 2022-2023 test period.
test_openet_cal = test_openet[
    (test_openet["date"] >= CAL_START) & (test_openet["date"] <= CAL_END)
]

test_obj_fn = make_site_objective(test_climate, test_openet_cal, test_params, OBJECTIVE)
 
# Coarse grid search for preview
coarse_vals = np.arange(COARSE_LO, COARSE_HI + 0.01, COARSE_STEP)
coarse_obj  = [test_obj_fn(v) for v in coarse_vals]
best_coarse_idx = int(np.argmin(coarse_obj))
 
# Response curve plot
# Response curve plot
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
ax.plot(coarse_vals, coarse_obj, color="steelblue", linewidth=2,
        marker="o", markersize=5)
ax.axvline(PET_MULT, color="grey", linestyle=":", linewidth=1,
           label=f"Default PET_mult={PET_MULT}")
ax.axvline(coarse_vals[best_coarse_idx], color="tomato",
           linestyle="--", linewidth=1.5,
           label=f"Best coarse = {coarse_vals[best_coarse_idx]:.2f}")
ax.set_xlabel("PET_mult", fontsize=10)
ax.set_ylabel(f"{OBJECTIVE}", fontsize=10)
ax.set_title(f"PET_mult sensitivity — {TEST_SITE} (vs OpenET 4 km gridcell)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(f"../Data/open_et/pet_mult_sensitivity_{TEST_SITE}_{OBJECTIVE}.png",
            dpi=150, bbox_inches="tight")
plt.show()
 
# Fine optimisation for preview site
_lo = max(0.1, coarse_vals[best_coarse_idx] - COARSE_STEP)
_hi = min(MAX_MULT, coarse_vals[best_coarse_idx] + COARSE_STEP)
test_opt = minimize_scalar(test_obj_fn, bounds=(_lo, _hi),
                            method="bounded", options={"xatol": 0.01})
best_pet_mult = round(test_opt.x, 3)
 
print(f"  Default objective : {test_obj_fn(PET_MULT):.3f}")
print(f"  Optimal PET_mult  : {best_pet_mult}")
print(f"  Optimal objective : {test_opt.fun:.3f}")
 
# Before/after time series for preview site
daily_default = run_wbm_with_spinup(test_climate, test_params, PET_MULT)
daily_opt     = run_wbm_with_spinup(test_climate, test_params, best_pet_mult)
scale         = 25.4 if TO_INCHES else 1.0
 
def _to_monthly_aet(daily):
    return (
        daily
        .assign(
            date_m = lambda d:
                pd.to_datetime(d["date"].dt.to_period("M").dt.to_timestamp()),
            AET_mm = lambda d: d["AET"] * scale,
        )
        .groupby("date_m", as_index=False)["AET_mm"].sum()
        .rename(columns={"date_m": "date"})
    )
 
monthly_default = _to_monthly_aet(daily_default)
monthly_opt     = _to_monthly_aet(daily_opt)
 
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)

# Gridcell OpenET ensemble — dark green solid (no ribbon)
ax.plot(test_openet["date"], test_openet[ENSEMBLE_COL],
        color="#1a5e2a", linewidth=1.5, linestyle="-",
        label="OpenET ensemble (4 km gridcell)", zorder=6)

# Flux tower
flux_sub = test_openet.dropna(subset=["aet_flux_mm"])
if not flux_sub.empty:
    ax.scatter(flux_sub["date"], flux_sub["aet_flux_mm"],
               color="crimson", s=25, zorder=8, label="Flux tower ET")

# WBM default — black dotted
ax.plot(monthly_default["date"], monthly_default["AET_mm"],
        color="black", linewidth=1, linestyle=":",
        alpha=0.8, zorder=4, label=f"WBM (PET_mult={PET_MULT})")

# WBM optimised — black solid
ax.plot(monthly_opt["date"], monthly_opt["AET_mm"],
        color="black", linewidth=1.8, linestyle="-",
        zorder=7, label=f"WBM (PET_mult={best_pet_mult}  obj={OBJECTIVE})")

ax.set_title(f"{TEST_SITE} — before vs optimised PET_mult ({OBJECTIVE})",
             fontsize=11, fontweight="bold")
ax.set_ylabel("ET (mm / month)", fontsize=10)
ax.set_ylim(bottom=0)
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.savefig(f"../Data/open_et/pet_mult_preview_{TEST_SITE}_{OBJECTIVE}.png",
            dpi=150, bbox_inches="tight")
plt.show()
 
# ── B4: All-sites optimisation (incremental — only missing sites) ────────────
print(f"\n── All-sites optimisation  objective='{OBJECTIVE}' ──────────────────")

# NOTE: cache filenames now say "site_cal" (not just OBJECTIVE) -- the
# multiplier here is fit on the CAL_START-CAL_END period only, a different
# (and not comparable) quantity from any pre-existing full-period-fit cache
# from before this change. Using a distinct filename means the incremental
# cache can't accidentally treat an old full-period fit as "already done".
PET_MULT_CACHE    = f"../Data/gridmet_cache/pet_mult_site_{OBJECTIVE}.csv"
WBM_OPT_CACHE     = f"../Data/gridmet_cache/wbm_results_site_cal_{OBJECTIVE}.csv"
WBM_MONTHLY_CACHE = f"../Data/gridmet_cache/wbm_monthly_site_cal_{OBJECTIVE}.csv"

all_calib_sites = point_params_df["site"].tolist()

# The pet_mult cache is the source of truth for "already calibrated" -- it's
# written together with the wbm_results_opt cache below, so the two stay in
# sync as long as this cell is the only thing that writes them.
existing_pet_mult_df, missing_calib_sites = wbm.load_and_filter_missing(
    all_sites=all_calib_sites,
    cache_path=PET_MULT_CACHE,
    site_col="site",
)
existing_wbm_opt_df, _ = wbm.load_and_filter_missing(
    all_sites=all_calib_sites,
    cache_path=WBM_OPT_CACHE,
    site_col="site",
    parse_dates=["date"],
)

if not missing_calib_sites:
    print(f"All {len(all_calib_sites)} sites already calibrated in {PET_MULT_CACHE} "
          f"-- skipping optimisation.")
    pet_mult_df     = existing_pet_mult_df
    wbm_results_opt = existing_wbm_opt_df

else:
    print(f"Calibrating {len(missing_calib_sites)} new site(s): {missing_calib_sites}")
    print(f"({len(all_calib_sites) - len(missing_calib_sites)} site(s) already "
          f"cached, skipped.)")

    pet_mult_results = []
    wbm_frames_opt   = []

    for row in point_params_df.itertuples():
        site = row.site
        if site not in missing_calib_sites:
            continue

        site_climate = climate_gee[climate_gee["site"] == site].copy()
        # Use gridcell-averaged OpenET for the WBM optimization objective,
        # restricted to the calibration period only (site_climate itself stays
        # full-period so the WBM re-run below covers 2016-2023 for later
        # held-out-period evaluation in notebook 07).
        site_openet_cal = (
            combined[combined["site"] == site]
            [["date", GRIDCELL_ENSEMBLE_COL]]
            .dropna(subset=[GRIDCELL_ENSEMBLE_COL])
            .rename(columns={GRIDCELL_ENSEMBLE_COL: ENSEMBLE_COL})
        )
        site_openet_cal = site_openet_cal[
            (site_openet_cal["date"] >= CAL_START) & (site_openet_cal["date"] <= CAL_END)
        ]

        if site_climate.empty or site_openet_cal.empty:
            print(f"  ⚠️  {site:<12} — skipping (no climate or calibration-period OpenET data)")
            pet_mult_results.append({
                "site": site, "pet_mult_opt": PET_MULT,
                "obj_default": np.nan, "obj_opt": np.nan,
                "improvement": np.nan, "status": "skipped",
            })
            continue

        print(f"  {site:<12} ...", end=" ")
        obj_fn = make_site_objective(site_climate, site_openet_cal, row, OBJECTIVE)

        # Coarse search
        c_vals  = np.arange(COARSE_LO, COARSE_HI + 0.01, COARSE_STEP)
        c_obj   = [obj_fn(v) for v in c_vals]
        best_i  = int(np.argmin(c_obj))
        lo      = max(0.1, c_vals[best_i] - COARSE_STEP)
        hi      = min(MAX_MULT, c_vals[best_i] + COARSE_STEP)

        # Fine search
        opt = minimize_scalar(obj_fn, bounds=(lo, hi),
                              method="bounded", options={"xatol": 0.01})

        obj_def      = round(obj_fn(PET_MULT), 3)
        obj_opt      = round(opt.fun, 3)
        pet_mult_opt = round(opt.x, 3)
        improvement  = round(obj_def - obj_opt, 3)

        pet_mult_results.append({
            "site":         site,
            "pet_mult_opt": pet_mult_opt,
            "obj_default":  obj_def,
            "obj_opt":      obj_opt,
            "improvement":  improvement,
            "status":       "optimised",
        })
        print(f"PET_mult={pet_mult_opt}  "
              f"{OBJECTIVE}: {obj_def:.3f}→{obj_opt:.3f}  (Δ={improvement:+.3f})")

        # Re-run WBM with optimal PET_mult
        try:
            daily = run_wbm_with_spinup(site_climate, row, pet_mult_opt)
            daily["site"]         = site
            daily["ecosystem"]    = row.ecosystem
            daily["state"]        = row.state
            daily["pet_mult_opt"] = pet_mult_opt
            wbm_frames_opt.append(daily)
        except Exception as e:
            print(f"    ⚠️  WBM re-run failed: {e}")

    # ── B5: Assemble new-site results and merge with cache ───────────────────────
    new_pet_mult_df = pd.DataFrame(pet_mult_results)

    print(f"\n── Optimisation summary ({OBJECTIVE}) — new sites ────────────────────")
    print(new_pet_mult_df.to_string(index=False))

    if not wbm_frames_opt:
        raise RuntimeError(
            "WBM re-run produced no results for any missing site — check "
            "site_climate/site_openet availability above."
        )

    new_wbm_opt_df = (
        pd.concat(wbm_frames_opt, ignore_index=True)
        .sort_values(["site", "date"])
        .reset_index(drop=True)
    )

    pet_mult_df = wbm.merge_and_save_cache(
        cache_path=PET_MULT_CACHE,
        existing_df=existing_pet_mult_df,
        new_rows_df=new_pet_mult_df,
        sort_cols=["site"],
    )
    wbm_results_opt = wbm.merge_and_save_cache(
        cache_path=WBM_OPT_CACHE,
        existing_df=existing_wbm_opt_df,
        new_rows_df=new_wbm_opt_df,
        sort_cols=["site", "date"],
    )
    print(f"\nSaved to {PET_MULT_CACHE} and {WBM_OPT_CACHE}")

print(f"\nMean PET_mult  : {pet_mult_df['pet_mult_opt'].mean():.3f}")
print(f"Range          : {pet_mult_df['pet_mult_opt'].min():.3f} – "
      f"{pet_mult_df['pet_mult_opt'].max():.3f}")
print(f"Sites improved : "
      f"{(pet_mult_df['improvement'] > 0).sum()} / {len(pet_mult_df)}")

# wbm_monthly_opt is rebuilt from the full (cached + new) wbm_results_opt --
# cheap local aggregation, no reason to cache this one separately.
wbm_monthly_opt = make_wbm_monthly(wbm_results_opt)

print(f"\nwbm_monthly_opt shape : {wbm_monthly_opt.shape}")
print(f"AET range (mm)        : {wbm_monthly_opt['aet_mm'].min():.1f} – "
      f"{wbm_monthly_opt['aet_mm'].max():.1f}")

wbm_monthly_opt.to_csv(WBM_MONTHLY_CACHE, index=False)
print(f"\nSaved to {WBM_MONTHLY_CACHE}")

# ═══════════════════════════════════════════════════════════════════════════════
# B4b: Per-ecosystem optimisation (single PET_mult per ecosystem, pooled fit)
# ═══════════════════════════════════════════════════════════════════════════════
# Rather than one PET_mult per site (B4 above), this fits ONE PET_mult per
# ecosystem category by pooling every site's matched (site-month,
# calibration-period-only) observations into a single objective score before
# optimizing. The idea: test whether ecosystem-level calibration captures
# most of per-site calibration's benefit while generalizing better to sites
# the fit didn't see directly. Notebook 07 does the head-to-head comparison
# of this against per-site-calibrated Oudin, default (uncalibrated) Oudin,
# and default Penman-Monteith, evaluated only on the held-out TEST period.

print(f"\n── Per-ecosystem optimisation  objective='{OBJECTIVE}' ──────────────────")

PET_MULT_ECO_CACHE    = f"../Data/gridmet_cache/pet_mult_ecosystem_{OBJECTIVE}.csv"
WBM_ECO_CACHE         = f"../Data/gridmet_cache/wbm_results_ecosystem_cal_{OBJECTIVE}.csv"
WBM_MONTHLY_ECO_CACHE = f"../Data/gridmet_cache/wbm_monthly_ecosystem_cal_{OBJECTIVE}.csv"

ecosystems_all = sorted(point_params_df["ecosystem"].dropna().unique())

existing_pet_mult_eco_df, missing_ecosystems = wbm.load_and_filter_missing(
    all_sites=ecosystems_all, cache_path=PET_MULT_ECO_CACHE, site_col="ecosystem",
)
existing_wbm_eco_df, _ = wbm.load_and_filter_missing(
    all_sites=ecosystems_all, cache_path=WBM_ECO_CACHE, site_col="ecosystem",
    parse_dates=["date"],
)


def make_ecosystem_objective(eco_climate_by_site, eco_openet_cal_by_site,
                              eco_params_by_site, obj_name):
    """
    Build an objective function that pools (obs, pred) pairs across every
    site in one ecosystem before scoring -- so a single PET_mult is optimized
    against the ecosystem's combined calibration-period record, not any one
    site's alone.
    """
    obj_fn = OBJECTIVE_FNS[obj_name]

    def _obj(pet_mult_val):
        obs_all, pred_all = [], []
        for site, site_climate in eco_climate_by_site.items():
            site_openet_cal = eco_openet_cal_by_site[site]
            params_row = eco_params_by_site[site]
            try:
                daily = run_wbm_with_spinup(site_climate, params_row, pet_mult_val)
                scale = 25.4 if TO_INCHES else 1.0
                monthly = (
                    daily.assign(
                        date_m=lambda d: pd.to_datetime(d["date"]).dt.to_period("M").dt.to_timestamp(),
                        AET_mm=lambda d: d["AET"] * scale,
                    )
                    .groupby("date_m", as_index=False)["AET_mm"].sum()
                    .rename(columns={"date_m": "date"})
                )
                merged = site_openet_cal.merge(monthly, on="date", how="inner")
                if merged.empty:
                    continue
                obs_all.append(merged[ENSEMBLE_COL].values)
                pred_all.append(merged["AET_mm"].values)
            except Exception:
                continue
        if not obs_all or sum(len(a) for a in obs_all) < 6:
            return 9999.0
        obs = np.concatenate(obs_all)
        pred = np.concatenate(pred_all)
        return obj_fn(obs, pred)

    return _obj


if not missing_ecosystems:
    pet_mult_eco_df = existing_pet_mult_eco_df
    wbm_results_eco = existing_wbm_eco_df
else:
    pet_mult_eco_results, wbm_frames_eco = [], []

    for eco in missing_ecosystems:
        eco_sites_df = point_params_df[point_params_df["ecosystem"] == eco]

        eco_climate_by_site, eco_openet_cal_by_site, eco_params_by_site = {}, {}, {}
        for row in eco_sites_df.itertuples():
            site = row.site
            site_climate = climate_gee[climate_gee["site"] == site].copy()
            site_openet_cal = (
                combined[combined["site"] == site]
                [["date", GRIDCELL_ENSEMBLE_COL]]
                .dropna(subset=[GRIDCELL_ENSEMBLE_COL])
                .rename(columns={GRIDCELL_ENSEMBLE_COL: ENSEMBLE_COL})
            )
            site_openet_cal = site_openet_cal[
                (site_openet_cal["date"] >= CAL_START) & (site_openet_cal["date"] <= CAL_END)
            ]
            if site_climate.empty or site_openet_cal.empty:
                continue
            eco_climate_by_site[site] = site_climate
            eco_openet_cal_by_site[site] = site_openet_cal
            eco_params_by_site[site] = row

        if not eco_climate_by_site:
            print(f"  ⚠️  {eco:<25} — skipping (no usable sites)")
            pet_mult_eco_results.append({
                "ecosystem": eco, "n_sites": 0, "pet_mult_opt": PET_MULT,
                "obj_default": np.nan, "obj_opt": np.nan,
                "improvement": np.nan, "status": "skipped",
            })
            continue

        print(f"  {eco:<25} ({len(eco_climate_by_site)} site(s)) ...", end=" ")
        obj_fn = make_ecosystem_objective(
            eco_climate_by_site, eco_openet_cal_by_site, eco_params_by_site, OBJECTIVE
        )

        c_vals = np.arange(COARSE_LO, COARSE_HI + 0.01, COARSE_STEP)
        c_obj  = [obj_fn(v) for v in c_vals]
        best_i = int(np.argmin(c_obj))
        lo = max(0.1, c_vals[best_i] - COARSE_STEP)
        hi = min(MAX_MULT, c_vals[best_i] + COARSE_STEP)

        opt = minimize_scalar(obj_fn, bounds=(lo, hi), method="bounded", options={"xatol": 0.01})

        obj_def      = round(obj_fn(PET_MULT), 3)
        obj_opt      = round(opt.fun, 3)
        pet_mult_opt = round(opt.x, 3)
        improvement  = round(obj_def - obj_opt, 3)

        pet_mult_eco_results.append({
            "ecosystem":    eco,
            "n_sites":      len(eco_climate_by_site),
            "pet_mult_opt": pet_mult_opt,
            "obj_default":  obj_def,
            "obj_opt":      obj_opt,
            "improvement":  improvement,
            "status":       "optimised",
        })
        print(f"PET_mult={pet_mult_opt}  {OBJECTIVE}: {obj_def:.3f}→{obj_opt:.3f}  (Δ={improvement:+.3f})")

        # Re-run WBM (full period, for continuity) for every site in this
        # ecosystem using the ecosystem's fitted multiplier.
        for site, site_climate in eco_climate_by_site.items():
            params_row = eco_params_by_site[site]
            try:
                daily = run_wbm_with_spinup(site_climate, params_row, pet_mult_opt)
                daily["site"]         = site
                daily["ecosystem"]    = eco
                daily["state"]        = params_row.state
                daily["pet_mult_opt"] = pet_mult_opt
                wbm_frames_eco.append(daily)
            except Exception as e:
                print(f"    ⚠️  WBM re-run failed for {site}: {e}")

    new_pet_mult_eco_df = pd.DataFrame(pet_mult_eco_results)

    print(f"\n── Ecosystem optimisation summary ({OBJECTIVE}) — new ecosystems ────────")
    print(new_pet_mult_eco_df.to_string(index=False))

    if not wbm_frames_eco:
        raise RuntimeError(
            "WBM re-run produced no results for any missing ecosystem — check "
            "eco_climate_by_site/eco_openet_cal_by_site availability above."
        )

    new_wbm_eco_df = (
        pd.concat(wbm_frames_eco, ignore_index=True)
        .sort_values(["site", "date"])
        .reset_index(drop=True)
    )

    # Safeguard: a site's ecosystem membership doesn't change, so this is
    # mostly precautionary -- but if it ever did, drop any existing cached
    # rows for sites now being recalibrated under a *different* ecosystem
    # grouping before merging, so we don't end up with duplicate site/date
    # rows fit under two different ecosystem multipliers.
    missing_eco_sites = point_params_df[
        point_params_df["ecosystem"].isin(missing_ecosystems)
    ]["site"].tolist()
    if existing_wbm_eco_df is not None and not existing_wbm_eco_df.empty:
        existing_wbm_eco_df = existing_wbm_eco_df[
            ~existing_wbm_eco_df["site"].isin(missing_eco_sites)
        ]

    pet_mult_eco_df = wbm.merge_and_save_cache(
        cache_path=PET_MULT_ECO_CACHE,
        existing_df=existing_pet_mult_eco_df,
        new_rows_df=new_pet_mult_eco_df,
        sort_cols=["ecosystem"],
    )
    wbm_results_eco = wbm.merge_and_save_cache(
        cache_path=WBM_ECO_CACHE,
        existing_df=existing_wbm_eco_df,
        new_rows_df=new_wbm_eco_df,
        sort_cols=["site", "date"],
    )
    print(f"\nSaved to {PET_MULT_ECO_CACHE} and {WBM_ECO_CACHE}")

print(f"\nEcosystems fit : {len(pet_mult_eco_df)}")
print(pet_mult_eco_df.to_string(index=False))

# wbm_monthly_eco is rebuilt from the full (cached + new) wbm_results_eco --
# cheap local aggregation, no reason to cache this one separately.
wbm_monthly_eco = make_wbm_monthly(wbm_results_eco)
wbm_monthly_eco.to_csv(WBM_MONTHLY_ECO_CACHE, index=False)

print(f"\nwbm_monthly_eco shape : {wbm_monthly_eco.shape}")
print(f"Saved to {WBM_MONTHLY_ECO_CACHE}")


Define new optimization parameters and perform again.

After the PET multiplier optimisation has run for all sites, we rebuild `combined_mm` from scratch using `build_combined_mm` so that the default and optimised WBM time series are both available in a single, clean DataFrame. All per-site metrics are then recomputed on this unified dataset to ensure comparisons are made on identical date ranges.

In [ ]:
# Always run this cell after any WBM re-run before any plotting or metrics.
# Rebuilds combined_mm cleanly from source DataFrames — no incremental merges.
# ═══════════════════════════════════════════════════════════════════════════════
 
# Pass all WBM variants you want in the plots.
# Keys become column names in combined_mm.
# wbm_monthly      = default PET_mult (already defined upstream)
# wbm_monthly_opt  = optimised PET_mult (produced by Cell B)
# ─── Build combined_mm with both point OpenET (flux comparison)
# and gridcell OpenET (WBM comparison) in one call.
combined_mm = build_combined_mm(
    openet_df           = openet_df,
    openet_gridcell_df  = openet_gridcell_df,
    flux_monthly = flux_monthly,
    wbm_monthly_dict = {
        "aet_wbm_mm":  wbm_monthly,      # default
        "aet_opt_mm":  wbm_monthly_opt,  # optimised (peak_nse or other)
    },
)
 
# Verify availability
availability = (
    combined_mm.groupby("site")
    .agg(
        n_openet     = (ENSEMBLE_COL,          "count"),
        n_openet_gc  = (GRIDCELL_ENSEMBLE_COL, "count"),
        n_wbm     = ("aet_wbm_mm",   "count"),
        n_opt     = ("aet_opt_mm",   "count"),
        n_flux    = ("aet_flux_mm",  "count"),
    )
    .reset_index()
)
 

# ── Gridcell OpenET column names (WBM vs OpenET comparisons) ─────────────────
GRIDCELL_ENSEMBLE_COL = "et_ensemble_gc_mm"
GRIDCELL_MIN_COL      = "et_ensemble_gc_min_mm"
GRIDCELL_MAX_COL      = "et_ensemble_gc_max_mm"

print("Data availability per site (combined_mm rebuilt from scratch):")
print(availability.to_string(index=False))
 
# Warn on any unexpected zeros
for col in ["n_openet", "n_wbm"]:
    zeros = availability[availability[col] == 0]["site"].tolist()
    if zeros:
        print(f"\n⚠️  Sites with 0 rows for {col}: {zeros}")
 
# Sites valid for plotting
sites_to_plot = availability[
    (availability["n_openet"] > 0) & (availability["n_wbm"] > 0)
]["site"].tolist()
 
print(f"\nSites available for plotting : {len(sites_to_plot)}")
print(f"  {sorted(sites_to_plot)}")
 
# Recompute all metrics on clean combined_mm
metrics_wbm_vs_openet = compute_metrics(
    combined_mm, obs_col=GRIDCELL_ENSEMBLE_COL, pred_col="aet_wbm_mm")
metrics_opt_vs_openet = compute_metrics(
    combined_mm, obs_col=GRIDCELL_ENSEMBLE_COL, pred_col="aet_opt_mm")
metrics_wbm_vs_flux   = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col="aet_wbm_mm")
metrics_opt_vs_flux   = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col="aet_opt_mm")
metrics_openet_vs_flux = compute_metrics(
    combined_mm, obs_col="aet_flux_mm", pred_col=ENSEMBLE_COL)
 
print("\n── Aggregate metrics vs OpenET ───────────────────────────────────────")
for label, mdf in [("Default WBM",  metrics_wbm_vs_openet),
                    (f"Optimised ({OBJECTIVE})", metrics_opt_vs_openet)]:
    valid = mdf[mdf["note"] == ""]
    print(f"\n  {label}:")
    for m in ["slope", "MBE", "MAE", "RMSE", "R2"]:
        print(f"    {m:<5} = {valid[m].mean():>8.3f}")
 
print("\n── Aggregate metrics vs Flux tower ───────────────────────────────────")
for label, mdf in [("OpenET",       metrics_openet_vs_flux),
                    ("Default WBM",  metrics_wbm_vs_flux),
                    (f"Optimised ({OBJECTIVE})", metrics_opt_vs_flux)]:
    valid = mdf[mdf["note"] == ""]
    if valid.empty:
        continue
    print(f"\n  {label}:")
    for m in ["slope", "MBE", "MAE", "RMSE", "R2"]:
        print(f"    {m:<5} = {valid[m].mean():>8.3f}")

In [ ]:
# ── Optimisation Summary — Performance Change & PET Multipliers ───────────────

# ── Gather per-site metrics before and after optimisation ─────────────────────
summary = (
    metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]
    [["site", "slope", "MBE", "MAE", "RMSE", "R2"]]
    .rename(columns={"slope": "slope_def", "MBE": "MBE_def",
                     "MAE": "MAE_def", "RMSE": "RMSE_def", "R2": "R2_def"})
    .merge(
        metrics_opt_vs_openet[metrics_opt_vs_openet["note"] == ""]
        [["site", "slope", "MBE", "MAE", "RMSE", "R2"]]
        .rename(columns={"slope": "slope_opt", "MBE": "MBE_opt",
                         "MAE": "MAE_opt", "RMSE": "RMSE_opt", "R2": "R2_opt"}),
        on="site", how="inner"
    )
    .merge(pet_mult_df[["site", "pet_mult_opt", "improvement", "status"]],
           on="site", how="left")
    .merge(flux_towers[["site", "x", "y", "ecosystem"]], on="site", how="left")
)

summary["RMSE_change"] = summary["RMSE_opt"] - summary["RMSE_def"]
summary["R2_change"]   = summary["R2_opt"]   - summary["R2_def"]
summary["MAE_change"]  = summary["MAE_opt"]  - summary["MAE_def"]

print(f"Sites in summary: {len(summary)}")
print(summary[["site", "pet_mult_opt", "RMSE_def", "RMSE_opt",
               "R2_def", "R2_opt", "RMSE_change"]].to_string(index=False))

# ── Figure layout: 2 rows × 3 cols ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(8, 6), constrained_layout=True)

# ── Panel 1: PET multiplier range, coloured by RMSE improvement ───────────────
ax = axes[0, 0]
sc = ax.scatter(
    summary["pet_mult_opt"],
    summary["RMSE_change"],
    c     = summary["RMSE_change"],
    cmap  = "RdYlGn_r",
    s     = 80,
    edgecolors = "k",
    linewidths = 0.5,
    zorder = 3,
)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.axvline(PET_MULT, color="steelblue", linestyle=":", linewidth=1,
           label=f"Default k={PET_MULT}")
#for _, row in summary.iterrows():
#    ax.annotate(row["site"], (row["pet_mult_opt"], row["RMSE_change"]),
#                fontsize=5, xytext=(4, 2), textcoords="offset points", color="black")
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("ΔRMSE (mm/month)", fontsize=8)
cbar.ax.tick_params(labelsize=7)
ax.set_xlabel("Optimised PET_mult (k)", fontsize=9)
ax.set_ylabel("ΔRMSE (opt − default, mm/month)", fontsize=9)
ax.set_title("PET multiplier vs RMSE improvement\n(negative = improved)", fontsize=9, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel 2: PET multiplier distribution by ecosystem ─────────────────────────
ax = axes[0, 1]
ecosystems = sorted(summary["ecosystem"].dropna().unique())
eco_colors = plt.cm.tab10.colors
eco_color_map = {e: eco_colors[i % 10] for i, e in enumerate(ecosystems)}

for eco in ecosystems:
    sub = summary[summary["ecosystem"] == eco]
    ax.scatter(
        [eco] * len(sub),
        sub["pet_mult_opt"],
        color = eco_color_map[eco],
        s = 60, edgecolors="k", linewidths=0.5,
        label=eco, zorder=3,
    )
    #for _, row in sub.iterrows():
    #    ax.annotate(row["site"], (eco, row["pet_mult_opt"]),
    #                fontsize=4.5, xytext=(4, 0),
    #                textcoords="offset points", color="grey")

ax.axhline(PET_MULT, color="steelblue", linestyle=":", linewidth=1,
           label=f"Default k={PET_MULT}")
ax.set_ylabel("Optimised PET_mult (k)", fontsize=9)
ax.set_title("PET multiplier by ecosystem", fontsize=9, fontweight="bold")
ax.tick_params(axis="x", labelsize=7, rotation=30)
ax.tick_params(axis="y", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)

# ── Panel 3: Before/after RMSE box plots ──────────────────────────────────────
ax = axes[0, 2]
bp = ax.boxplot(
    [summary["RMSE_def"], summary["RMSE_opt"]],
    tick_labels=["Default WBM", f"Optimised\n({OBJECTIVE})"],
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
bp["boxes"][0].set_facecolor("#b0c4de")
bp["boxes"][1].set_facecolor("#90c990")

# Overlay individual site points with connecting lines
for _, row in summary.iterrows():
    ax.plot([1, 2], [row["RMSE_def"], row["RMSE_opt"]],
            color="grey", linewidth=0.6, alpha=0.5, zorder=2)
    ax.scatter([1, 2], [row["RMSE_def"], row["RMSE_opt"]],
               color="grey", s=15, zorder=3, alpha=0.7)

ax.set_ylabel("RMSE (mm/month)", fontsize=9)
ax.set_title("RMSE before vs after optimisation", fontsize=9, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel 4: Before/after R² box plots ───────────────────────────────────────
ax = axes[1, 0]
bp2 = ax.boxplot(
    [summary["R2_def"], summary["R2_opt"]],
    tick_labels=["Default WBM", f"Optimised\n({OBJECTIVE})"],
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
bp2["boxes"][0].set_facecolor("#b0c4de")
bp2["boxes"][1].set_facecolor("#90c990")

for _, row in summary.iterrows():
    ax.plot([1, 2], [row["R2_def"], row["R2_opt"]],
            color="grey", linewidth=0.6, alpha=0.5, zorder=2)
    ax.scatter([1, 2], [row["R2_def"], row["R2_opt"]],
               color="grey", s=15, zorder=3, alpha=0.7)

ax.axhline(0, color="tomato", linestyle="--", linewidth=1, label="R²=0")
ax.set_ylabel("R²", fontsize=9)
ax.set_title("R² before vs after optimisation", fontsize=9, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel 5: Improvement potential — sites ranked by ΔRMSE ───────────────────
ax = axes[1, 1]
ranked = summary.sort_values("RMSE_change")
colors = ["#2e8b57" if v < 0 else "#cd5c5c" for v in ranked["RMSE_change"]]
bars = ax.barh(ranked["site"], ranked["RMSE_change"],
               color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("ΔRMSE (mm/month)", fontsize=9)
ax.set_title("RMSE change per site\n(negative = improved)", fontsize=9, fontweight="bold")
ax.tick_params(axis="y", labelsize=6.5)
ax.tick_params(axis="x", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)

# ── Panel 6: PET multiplier histogram ────────────────────────────────────────
ax = axes[1, 2]
ax.hist(summary["pet_mult_opt"], bins=10, color="steelblue",
        edgecolor="white", linewidth=0.8, alpha=0.85)
ax.axvline(PET_MULT, color="tomato", linestyle="--", linewidth=1.5,
           label=f"Default k={PET_MULT}")
ax.axvline(summary["pet_mult_opt"].median(), color="black",
           linestyle="-", linewidth=1.5,
           label=f"Median k={summary['pet_mult_opt'].median():.2f}")
ax.set_xlabel("Optimised PET_mult (k)", fontsize=9)
ax.set_ylabel("Number of sites", fontsize=9)
ax.set_title("Distribution of optimised PET multipliers", fontsize=9, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)

fig.suptitle(
    f"Optimisation Summary — PET_mult ({OBJECTIVE}) vs OpenET 4 km gridcell\n"
    f"n={len(summary)} sites",
    fontsize=12, fontweight="bold",
)

plt.savefig(f"../Data/open_et/optimisation_summary_{OBJECTIVE}.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to Data/open_et/optimisation_summary_{OBJECTIVE}.png")

### Optimised WBM — time series comparison

The faceted plot below shows each site's monthly ET from the default WBM, the PET-optimised WBM, OpenET ensemble, and the flux tower, allowing visual assessment of improvement after calibration.

In [ ]:
# ── Colour / style constants ──────────────────────────────────────────────────
ENSEMBLE_COL = "et_ensemble_mm"
MIN_COL      = "et_ensemble_min_mm"
MAX_COL      = "et_ensemble_max_mm"

# ── Merge gridcell ensemble into combined_mm for plotting ─────────────────────
gc_mean = (
    combined_gridcell_mm[["site", "date", ENSEMBLE_COL]]
    .rename(columns={ENSEMBLE_COL: "et_ensemble_gc_mm"})
    .dropna(subset=["et_ensemble_gc_mm"])
)

# Drop if already present from a previous run of this cell
if "et_ensemble_gc_mm" in combined_mm.columns:
    combined_mm = combined_mm.drop(columns=["et_ensemble_gc_mm"])

combined_mm = combined_mm.merge(gc_mean, on=["site", "date"], how="left")


# ── D1: Faceted time series ────────────────────────────────────────────────────
sites   = sorted(sites_to_plot)
n_sites = len(sites)
n_cols  = 4
n_rows  = int(np.ceil(n_sites / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(7, n_rows * 1.4),
    sharey=False, constrained_layout=True,
)
axes_flat = axes.flatten()

for ax, site in zip(axes_flat, sites):
    sub  = combined_mm[combined_mm["site"] == site].sort_values("date")
    m_d  = metrics_wbm_vs_openet[metrics_wbm_vs_openet["site"] == site]
    m_o  = metrics_opt_vs_openet[metrics_opt_vs_openet["site"] == site]
    mult = pet_mult_df[pet_mult_df["site"] == site]["pet_mult_opt"].iloc[0] \
           if site in pet_mult_df["site"].values else PET_MULT

    # Gridcell-averaged OpenET ensemble mean — dark green solid
    ax.plot(sub["date"], sub["et_ensemble_gc_mm"],
            color="#1a5e2a", linewidth=1.5, linestyle="-",
            zorder=5, label="OpenET ensemble (4 km gridcell)")

    # Default WBM — black dotted
    if "aet_wbm_mm" in sub.columns:
        ax.plot(sub["date"], sub["aet_wbm_mm"],
                color="black", linewidth=1, linestyle=":",
                alpha=0.8, zorder=4, label="WBM default")

    # Optimised WBM — black solid
    if "aet_opt_mm" in sub.columns:
        ax.plot(sub["date"], sub["aet_opt_mm"],
                color="black", linewidth=1.8,
                linestyle="-", zorder=6, label="WBM optimised")

    # Flux tower scatter
    flux_sub = sub.dropna(subset=["aet_flux_mm"])
    if not flux_sub.empty:
        ax.scatter(flux_sub["date"], flux_sub["aet_flux_mm"],
                   color="crimson", s=10, zorder=7, label="Flux tower ET")

    # Annotation
    r2_d   = m_d["R2"].iloc[0]   if not m_d.empty else np.nan
    r2_o   = m_o["R2"].iloc[0]   if not m_o.empty else np.nan
    rmse_d = m_d["RMSE"].iloc[0] if not m_d.empty else np.nan
    rmse_o = m_o["RMSE"].iloc[0] if not m_o.empty else np.nan
    ax.annotate(
        f"k={mult:.2f}\n"
        f"R²: {r2_d:.2f}→{r2_o:.2f}\n"
        f"RMSE: {rmse_d:.1f}→{rmse_o:.1f}",
        xy=(0.04, 0.97), xycoords="axes fraction",
        fontsize=6, va="top", color="steelblue",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                  alpha=0.75, edgecolor="none"),
    )

    ax.set_title(f"{site}", fontsize=9, fontweight="bold", pad=4)
    ax.set_ylabel("ET (mm / month)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

for ax in axes_flat[n_sites:]:
    ax.set_visible(False)

if n_sites < len(axes_flat):
    leg_ax = axes_flat[n_sites]
    leg_ax.set_visible(True)
    leg_ax.axis("off")
    leg_ax.legend(
        handles=[
            mlines.Line2D([], [], color="#1a5e2a", linewidth=1.5,
                          linestyle="-",
                          label="OpenET ensemble (4 km gridcell)"),
            mlines.Line2D([], [], color="black", linewidth=1,
                          linestyle=":", alpha=0.8, label="WBM default"),
            mlines.Line2D([], [], color="black", linewidth=1.8,
                          linestyle="-",
                          label=f"WBM optimised ({OBJECTIVE})"),
            mlines.Line2D([], [], color="crimson", linewidth=0,
                          marker="o", markersize=5, label="Flux tower ET"),
        ],
        loc="center", fontsize=8, frameon=False,
        title="Data sources", title_fontsize=9,
    )

fig.suptitle(
    f"Monthly ET — NPS WBM default vs {OBJECTIVE}-optimised vs OpenET (4 km gridcell)\n"
    f"k = PET_mult  |  R² and RMSE before→after",
    fontsize=11, fontweight="bold",
)
plt.savefig(f"../Data/open_et/et_comparison_{OBJECTIVE}_timeseries.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── D2: Scatter plots ─────────────────────────────────────────────────────────
combined_plot = combined_mm[combined_mm["site"].isin(sites_to_plot)].copy()
combined_gridcell_plot = combined_gridcell_mm[
    combined_gridcell_mm["site"].isin(sites_to_plot)
].copy()


# ── Merge aet_opt_mm into gridcell plot for optimised WBM comparisons ─────────
combined_gridcell_plot = combined_gridcell_plot.merge(
    combined_mm[["site", "date", "aet_opt_mm"]].dropna(subset=["aet_opt_mm"]),
    on=["site", "date"],
    how="left",
)

make_scatter_facets(
    metrics_df  = metrics_wbm_vs_openet,
    combined_df = combined_gridcell_plot,
    obs_col     = ENSEMBLE_COL,
    pred_col    = "aet_wbm_mm",
    obs_label   = "OpenET Ensemble — 4 km gridcell (mm/month)",
    pred_label  = "WBM AET default (mm/month)",
    title       = "WBM AET (default) vs OpenET Ensemble (4 km gridcell)",
    save_path   = "../Data/open_et/scatter_wbm_default_vs_openet.png",
)

make_scatter_facets(
    metrics_df  = metrics_opt_vs_openet,
    combined_df = combined_gridcell_plot,
    obs_col     = ENSEMBLE_COL,
    pred_col    = "aet_opt_mm",
    obs_label   = "OpenET Ensemble — 4 km gridcell (mm/month)",
    pred_label  = f"WBM AET {OBJECTIVE}-optimised (mm/month)",
    title       = f"WBM AET ({OBJECTIVE}-optimised) vs OpenET (Ens. grid)",
    save_path   = f"../Data/open_et/scatter_wbm_{OBJECTIVE}_vs_openet.png",
)

make_scatter_facets(
    metrics_df  = metrics_opt_vs_flux,
    combined_df = combined_plot,
    obs_col     = "aet_flux_mm",
    pred_col    = "aet_opt_mm",
    obs_label   = "Flux Tower ET (mm/month)",
    pred_label  = f"WBM AET {OBJECTIVE}-optimised (mm/month)",
    title       = f"WBM AET ({OBJECTIVE}-optimised) vs Flux Tower ET",
    save_path   = f"../Data/open_et/scatter_wbm_{OBJECTIVE}_vs_flux.png",
)

make_scatter_facets(
    metrics_df  = metrics_openet_vs_flux,
    combined_df = combined_plot,
    obs_col     = "aet_flux_mm",
    pred_col    = ENSEMBLE_COL,
    obs_label   = "Flux Tower ET (mm/month)",
    pred_label  = "OpenET Ensemble — point (mm/month)",
    title       = "OpenET Ensemble (point) vs Flux Tower ET",
    save_path   = "../Data/open_et/scatter_openet_vs_flux.png",
)

print("All plots saved to Data/open_et/")

Now, plot updated maps:

In [ ]:
# ── Step 1: Merge optimised metrics with tower coordinates ────────────────────
map_df = (
    metrics_opt_vs_openet[metrics_opt_vs_openet["note"] == ""]
    .merge(flux_towers[["site", "x", "y", "ecosystem"]], on="site", how="left")
    .merge(pet_mult_df[["site", "pet_mult_opt"]], on="site", how="left")
    .dropna(subset=["x", "y", "MBE", "RMSE"])
    .reset_index(drop=True)
)

print(f"Sites in map: {len(map_df)}")
print(map_df[["site", "x", "y", "ecosystem",
              "pet_mult_opt", "MBE", "RMSE", "R2"]].to_string(index=False))

# ── Step 2: Load state boundaries ─────────────────────────────────────────────
try:
    states = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2020/shp/"
        "cb_2020_us_state_20m.zip"
    )
    name_col = "NAME"
except Exception:
    states = gpd.read_file(
        "https://raw.githubusercontent.com/nvkelso/"
        "natural-earth-vector/master/geojson/"
        "ne_110m_admin_1_states_provinces.geojson"
    )
    name_col = "name"

NON_CONUS = ["Alaska", "Hawaii", "Puerto Rico", "Guam",
             "United States Virgin Islands", "American Samoa",
             "Commonwealth of the Northern Mariana Islands"]
states_conus = states[~states[name_col].isin(NON_CONUS)]

LON_MIN, LON_MAX = -125, -65
LAT_MIN, LAT_MAX =   24,  50

# ── Step 3: Colormaps ──────────────────────────────────────────────────────────
mbe_abs_max = np.ceil(map_df["MBE"].abs().max() / 10) * 10
mbe_norm    = mcolors.TwoSlopeNorm(vmin=-mbe_abs_max, vcenter=0,
                                    vmax=mbe_abs_max)
mbe_cmap    = "RdBu_r"

rmse_norm   = mcolors.Normalize(vmin=0,
                                 vmax=np.ceil(map_df["RMSE"].max() / 10) * 10)
rmse_cmap   = "YlOrRd"

# ── Step 4: Two-panel map ──────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(16, 6), constrained_layout=True
)

for ax, metric, cmap, norm, label in [
    (axes[0], "MBE",  mbe_cmap,  mbe_norm,
     "MBE (mm/month)\nnegative = WBM underestimates OpenET"),
    (axes[1], "RMSE", rmse_cmap, rmse_norm,
     "RMSE (mm/month)"),
]:
    # Basemap
    states_conus.plot(ax=ax, color="whitesmoke", edgecolor="grey",
                      linewidth=0.4, zorder=1)
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect("equal")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude",  fontsize=8)

    # Points — size proportional to magnitude
    vals  = map_df[metric].values
    sizes = (np.abs(vals) / np.abs(vals).max() * 300).clip(30)

    sc = ax.scatter(
        map_df["x"], map_df["y"],
        c=vals, s=sizes,
        cmap=cmap, norm=norm,
        zorder=5,
        edgecolors="k", linewidths=0.5,
        alpha=0.9,
    )

    # Site labels — show site ID, metric value, and PET_mult
    for _, row in map_df.iterrows():
        ax.annotate(
            f"{row['site']}\n"
            f"{metric}={row[metric]:.1f}\n"
            f"k={row['pet_mult_opt']:.2f}",
            xy=(row["x"], row["y"]),
            xytext=(6, 4), textcoords="offset points",
            fontsize=5.5, color="black", zorder=6,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                      alpha=0.6, edgecolor="none"),
        )

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02, aspect=20)
    cbar.set_label(label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    # Size legend in first empty corner
    ref_vals  = (
        [-100, -50, 50, 100] if metric == "MBE"
        else [25, 75, 150]
    )
    ref_vals  = [v for v in ref_vals if abs(v) <= np.abs(vals).max()]
    ref_sizes = [abs(v) / np.abs(vals).max() * 300 for v in ref_vals]
    size_handles = [
        plt.scatter([], [], s=s, color="grey",
                    edgecolors="k", linewidths=0.5,
                    alpha=0.7,
                    label=f"{v:+.0f}" if metric == "MBE" else f"{v:.0f}")
        for s, v in zip(ref_sizes, ref_vals)
    ]
    ax.legend(
        handles=size_handles,
        title=f"|{metric}| magnitude",
        title_fontsize=7, fontsize=7,
        loc="lower left", frameon=True, framealpha=0.8,
    )

    ax.set_title(
        f"WBM AET vs OpenET — {metric}\n"
        f"({OBJECTIVE}-optimised PET_mult)",
        fontsize=11, fontweight="bold", pad=8,
    )

fig.suptitle(
    f"WBM AET vs OpenET: spatial bias and error after {OBJECTIVE} calibration\n"
    "Point size ∝ magnitude  |  k = site-specific PET_mult",
    fontsize=12, fontweight="bold",
)

save_path = f"../Data/open_et/map_wbm_{OBJECTIVE}_bias_rmse.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

# ── Step 5: Before vs after comparison table ───────────────────────────────────
print("\n── MBE and RMSE: default vs optimised by site ───────────────────────────")
compare = (
    metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]
    [["site", "MBE", "RMSE", "R2"]]
    .rename(columns={"MBE": "MBE_default", "RMSE": "RMSE_default",
                     "R2": "R2_default"})
    .merge(
        metrics_opt_vs_openet[metrics_opt_vs_openet["note"] == ""]
        [["site", "MBE", "RMSE", "R2"]]
        .rename(columns={"MBE": "MBE_opt", "RMSE": "RMSE_opt",
                         "R2": "R2_opt"}),
        on="site", how="outer"
    )
    .merge(pet_mult_df[["site", "pet_mult_opt"]], on="site", how="left")
    .assign(
        ΔMBE  = lambda d: (d["MBE_opt"]  - d["MBE_default"]).round(1),
        ΔRMSE = lambda d: (d["RMSE_opt"] - d["RMSE_default"]).round(1),
        ΔR2   = lambda d: (d["R2_opt"]   - d["R2_default"]).round(3),
    )
    .sort_values("RMSE_opt")
)

print(compare[["site", "pet_mult_opt",
               "MBE_default", "MBE_opt",   "ΔMBE",
               "RMSE_default","RMSE_opt",  "ΔRMSE",
               "R2_default",  "R2_opt",    "ΔR2"]]
      .to_string(index=False))

print(f"\nMean RMSE: {compare['RMSE_default'].mean():.1f} → "
      f"{compare['RMSE_opt'].mean():.1f} mm/month "
      f"(Δ={compare['ΔRMSE'].mean():+.1f})")
print(f"Mean MBE : {compare['MBE_default'].mean():.1f} → "
      f"{compare['MBE_opt'].mean():.1f} mm/month "
      f"(Δ={compare['ΔMBE'].mean():+.1f})")
print(f"Mean R²  : {compare['R2_default'].mean():.3f} → "
      f"{compare['R2_opt'].mean():.3f} "
      f"(Δ={compare['ΔR2'].mean():+.3f})")

### Performrancee by ecosystem

The following shows performance by ecosystem.

In [ ]:
# Plot performance by ecosystem:

# ── Step 1: Build ecosystem metrics table ─────────────────────────────────────
# Merge all three comparison metrics with ecosystem info
eco_metrics = (
    metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]
    [["site", "RMSE", "MBE", "R2"]]
    .rename(columns={"RMSE": "RMSE_wbm_default",
                     "MBE":  "MBE_wbm_default",
                     "R2":   "R2_wbm_default"})
    .merge(
        metrics_opt_vs_openet[metrics_opt_vs_openet["note"] == ""]
        [["site", "RMSE", "MBE", "R2"]]
        .rename(columns={"RMSE": "RMSE_wbm_opt",
                         "MBE":  "MBE_wbm_opt",
                         "R2":   "R2_wbm_opt"}),
        on="site", how="outer"
    )
    .merge(
        metrics_openet_vs_flux[metrics_openet_vs_flux["note"] == ""]
        [["site", "RMSE", "MBE", "R2"]]
        .rename(columns={"RMSE": "RMSE_openet_flux",
                         "MBE":  "MBE_openet_flux",
                         "R2":   "R2_openet_flux"}),
        on="site", how="outer"
    )
    .merge(
        metrics_opt_vs_flux[metrics_opt_vs_flux["note"] == ""]
        [["site", "RMSE", "MBE", "R2"]]
        .rename(columns={"RMSE": "RMSE_wbm_opt_flux",
                         "MBE":  "MBE_wbm_opt_flux",
                         "R2":   "R2_wbm_opt_flux"}),
        on="site", how="outer"
    )
    .merge(
        flux_towers[["site", "ecosystem"]],
        on="site", how="left"
    )
    .merge(
        pet_mult_df[["site", "pet_mult_opt"]],
        on="site", how="left"
    )
)

print("Ecosystem metrics table:")
print(eco_metrics[["site", "ecosystem", "pet_mult_opt",
                    "RMSE_wbm_default", "RMSE_wbm_opt",
                    "RMSE_openet_flux", "RMSE_wbm_opt_flux"]]
      .to_string(index=False))

# ── Step 2: Define comparisons to plot ────────────────────────────────────────
COMPARISONS = {
    "WBM default\nvs OpenET":    ("RMSE_wbm_default", "darkblue",   ":"),
    f"WBM {OBJECTIVE}\nvs OpenET": ("RMSE_wbm_opt",   "steelblue",   "-"),
    "OpenET\nvs Flux tower":     ("RMSE_openet_flux",  "seagreen",   "-"),
    f"WBM {OBJECTIVE}\nvs Flux tower": ("RMSE_wbm_opt_flux","tomato","-"),
}

ecosystems = sorted(eco_metrics["ecosystem"].dropna().unique())
n_eco      = len(ecosystems)

# ── Step 3: Two-panel figure — strip chart + ecosystem bar chart ───────────────
fig, axes = plt.subplots(
    2, 1,
    figsize=(7, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 1]},
)

# ── Panel A: strip chart — RMSE per site, coloured by comparison ──────────────
ax = axes[0]
x_positions = {label: i for i, label in enumerate(COMPARISONS)}
jitter       = 0.12

for label, (col, color, ls) in COMPARISONS.items():
    x = x_positions[label]
    vals = eco_metrics[col].dropna()
    ax.scatter(
        np.full(len(vals), x) + np.random.uniform(-jitter, jitter, len(vals)),
        vals,
        color=color, alpha=0.7, s=50, zorder=4,
        linestyle=ls,
    )
    # Median bar
    ax.hlines(vals.median(), x - 0.25, x + 0.25,
              color=color, linewidth=2.5, zorder=5)
    # Label median
    ax.text(x, vals.median() + 1.5, f"{vals.median():.1f}",
            ha="center", fontsize=7, color="black", fontweight="bold")

ax.set_xticks(list(x_positions.values()))
ax.set_xticklabels(list(x_positions.keys()), fontsize=8)
ax.set_ylabel("RMSE (mm / month)", fontsize=9)
ax.set_title("RMSE by comparison type\n(bar = median, dots = individual sites)",
             fontsize=10, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)

# ── Panel B: grouped bar chart — mean RMSE per ecosystem ──────────────────────
ax = axes[1]

bar_cols   = [col for _, (col, _, _) in COMPARISONS.items()]
bar_labels = list(COMPARISONS.keys())
bar_colors = [color for _, (_, color, _) in COMPARISONS.items()]

n_comp   = len(COMPARISONS)
bar_w    = 0.18
x_eco    = np.arange(n_eco)

for i, (label, (col, color, ls)) in enumerate(COMPARISONS.items()):
    # Mean RMSE per ecosystem (skip NaN)
    eco_means = [
        eco_metrics[eco_metrics["ecosystem"] == eco][col].mean()
        for eco in ecosystems
    ]
    offset = (i - n_comp / 2 + 0.5) * bar_w
    bars = ax.bar(
        x_eco + offset, eco_means,
        width=bar_w, color=color, alpha=0.75,
        label=label.replace("\n", " "),
        edgecolor="white", linewidth=0.5,
    )
    # Value labels on bars
    for bar, val in zip(bars, eco_means):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.8,
                f"{val:.0f}",
                ha="center", va="bottom",
                fontsize=6, color="black",
            )

# Annotate n per ecosystem
for i, eco in enumerate(ecosystems):
    n = eco_metrics[eco_metrics["ecosystem"] == eco]["site"].nunique()
    ax.text(i, -4, f"n={n}", ha="center", fontsize=7, color="grey")

ax.set_xticks(x_eco)
ax.set_xticklabels(
    [e.replace(" ", "\n") for e in ecosystems],
    fontsize=8,
)
ax.set_ylabel("Mean RMSE (mm / month)", fontsize=9)
ax.set_title("Mean RMSE by ecosystem type",
             fontsize=10, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.legend(fontsize=7, frameon=False, loc="upper left")

ax.set_xticklabels(
    [e.replace(" ", "\n") for e in ecosystems],
    fontsize=8,
    rotation=30,
    ha="right",
)

fig.suptitle(
    f"ET model RMSE by ecosystem and comparison type",
    fontsize=11, fontweight="bold",
)

save_path = f"../Data/open_et/rmse_by_ecosystem_{OBJECTIVE}.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

# ── Step 4: Summary table — mean metrics per ecosystem ────────────────────────
print("\n── Mean metrics per ecosystem ───────────────────────────────────────────")
eco_summary = (
    eco_metrics
    .groupby("ecosystem")
    .agg(
        n_sites          = ("site",              "nunique"),
        pet_mult_mean    = ("pet_mult_opt",       "mean"),
        RMSE_wbm_default = ("RMSE_wbm_default",  "mean"),
        RMSE_wbm_opt     = ("RMSE_wbm_opt",      "mean"),
        RMSE_openet_flux = ("RMSE_openet_flux",  "mean"),
        RMSE_wbm_flux    = ("RMSE_wbm_opt_flux", "mean"),
        R2_wbm_default   = ("R2_wbm_default",    "mean"),
        R2_wbm_opt       = ("R2_wbm_opt",        "mean"),
    )
    .round(2)
    .reset_index()
)
print(eco_summary.to_string(index=False))

## Cache calibration outputs for notebook 05

Notebook 05 (figures/export) needs several objects this notebook computes in memory (`OBJECTIVE`, `eco_metrics`, and five `metrics_*_vs_*` DataFrames). Previously these weren't saved to disk, so notebook 05 only worked if it was run in the *same* live kernel as this notebook immediately after it -- fragile in practice, since opening a notebook file in Jupyter normally starts a fresh kernel with no memory of this one. This cell caches all of it to `Data/gridmet_cache/`, keyed by `OBJECTIVE`, so notebook 05 can reload it from disk in a fresh kernel.

In [ ]:
# ── Cache everything notebook 05 needs, so it no longer depends on
# sharing a live kernel with this notebook ────────────────────────────────────
import os
os.makedirs("../Data/gridmet_cache", exist_ok=True)

# OBJECTIVE itself isn't a DataFrame -- stash it in a small text file so
# notebook 05 knows which OBJECTIVE-suffixed files to load.
with open("../Data/gridmet_cache/last_objective.txt", "w") as f:
    f.write(OBJECTIVE)

_metrics_to_cache = {
    "metrics_wbm_vs_openet":   metrics_wbm_vs_openet,
    "metrics_opt_vs_openet":   metrics_opt_vs_openet,
    "metrics_wbm_vs_flux":     metrics_wbm_vs_flux,
    "metrics_opt_vs_flux":     metrics_opt_vs_flux,
    "metrics_openet_vs_flux":  metrics_openet_vs_flux,
    "metrics_cal_vs_openet":   metrics_cal_vs_openet,
    "metrics_cal_vs_flux":     metrics_cal_vs_flux,
    "eco_metrics":             eco_metrics,
}

for name, df in _metrics_to_cache.items():
    path = f"../Data/gridmet_cache/{name}_{OBJECTIVE}.csv"
    df.to_csv(path, index=False)

print(f"Cached OBJECTIVE='{OBJECTIVE}' and "
      f"{len(_metrics_to_cache)} metrics tables to Data/gridmet_cache/ "
      f"for notebook 05.")
